# GCN-BiGRU (AGB-Net) Intrusion Detection Pipeline

Unified notebook for **NSL-KDD**, **CICIDS2017**, **CIC-IoT-2023**, **Edge-IIoTset**,
and **TON_IoT**.

**Run All** executes the shared definitions and then the **Batch Runner** (Section 16), which
evaluates every dataset in `DATASETS_TO_RUN` back-to-back and fully unattended -- each in an
independent pass, each with its own hyperparameters.

To instead step through one dataset by hand, set `DATASET` and `RUN_SINGLE_DEMO = True` in the
config cell; that enables the linear walkthrough in Sections 2-15. The walkthrough is **off by
default** because the batch runner already recomputes every reported metric and every figure
cited in the paper, so leaving it on would train `DATASET` a second time for no extra result.

Every dataset uses the same preprocessing recipe (median imputation, log1p
feature augmentation, standardization, low-variance filtering) and the same GCN-BiGRU
(AGB-Net) architecture.

Pipeline overview:
1. Configuration
2. Load raw data (dataset-specific: files/columns/labels differ per benchmark)
3. Shared preprocessing: split -> impute -> log1p -> scale -> variance filter
4. Feature-correlation graph for the GCN front-end
5. Adaptive graph-convolution layer (learnable adjacency, Phase 3A) + adjacency comparison (RQ2)
6. GCN-BiGRU model definition + baseline architectures (BiGRU-only, GAT-BiGRU; RQ1)
7. Class weights
8. **Secretary Bird Optimization Algorithm (SBOA)** hyperparameter search (Phase 1)
9. Final model training at full epochs/full data with the SBOA-selected hyperparameters
10. Evaluation: performance table, confusion matrix, class-wise indicators, ROC/PR curves, timing
11. Extra experiments: re-implemented baselines (RQ1), fixed-vs-adaptive-graph ablation (RQ2),
    top-k neighbourhood-size sensitivity sweep (RQ3)
12. Batch runner: every dataset evaluated independently and unattended in one pass

Steps 2-10 are the `RUN_SINGLE_DEMO` walkthrough; step 12 re-derives all of them per dataset
via `run_pipeline()`.

Reproducibility scaffolding (Phase 0): GPU is used when available (with memory growth and
optional `mixed_float16`, falling back to CPU otherwise), all results -- including exact
train/val/test split sizes, confusion-matrix counts, per-class indicators, ROC/PR curve data,
adjacency-comparison statistics, and timing -- are persisted per dataset to
`results/results_<dataset>.json` (plus `results/baselines_<dataset>.json`,
`results/ablation_adaptive_<dataset>.json`, and `results/topk_sweep_<dataset>.json`) so tables
and plots can be rebuilt without re-running training, and package versions are printed for the
reproducibility statement.


**Description:** CONFIGURATION -- choose the dataset to run the pipeline on

In [1]:
DATASET = "cicids"  
SEED       = 42
EPOCHS     = 100
BATCH      = 512
TIMESTEPS  = 10
DROPOUT    = 0.3
REG        = 1e-3
TOP_K      = 8        
LR         = 1e-3
TEST_FRAC  = 0.15
VAL_FRAC   = 0.176     
SEEDS            = [42, 1, 7, 13, 21]  
NSL_NATIVE_SPLIT = False               
RUN_SBOA          = False
SBOA_SAMPLE_FRAC  = 0.15   
SBOA_POP          = 8      
SBOA_ITERS        = 12     
SBOA_EPOCHS       = 12     
FINAL_FULL_DATA   = True   
ADAPTIVE_GRAPH = True   
RUN_SINGLE_DEMO = False
DATASETS_ALL = ("nslkdd", "cicids", "ciciot23", "edge_iiot", "ton_iot")
DATASETS_TO_RUN = ["cicids"]
MAX_SAMPLES = {
    "ciciot23":  None,   
    "edge_iiot": None,   
    "ton_iot":   None,      
}
RUN_BASELINES          = True
BASELINE_DATASETS      = ["cicids", "nslkdd"]
RUN_ADAPTIVE_ABLATION  = False
ABLATION_DATASETS      = ["cicids", "nslkdd"]
RUN_TOPK_SWEEP         = False
SWEEP_DATASETS         = ["cicids", "nslkdd"]
TOPK_SWEEP_VALUES      = [2, 4, 6, 8, 10, 12]
TOPK_SWEEP_EPOCHS      = 25   
RUN_TIMESTEPS_SWEEP     = True
TIMESTEPS_SWEEP_DATASETS = ["nslkdd", "cicids", "ciciot23", "edge_iiot", "ton_iot"]
TIMESTEPS_SWEEP_VALUES  = [10, 14, 18, 22, 26, 30]
TIMESTEPS_SWEEP_EPOCHS  = 25   
assert DATASET in DATASETS_ALL, f"Unknown DATASET: {DATASET}"

**Description:** Non-interactive Agg backend: this notebook runs unattended (batch runner over 6

In [2]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["PYTHONHASHSEED"] = str(SEED)
import json as _json
import platform
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from scipy.ndimage import uniform_filter1d
from scipy.special import gamma as sp_gamma
import tensorflow as tf
GPUS = tf.config.list_physical_devices("GPU")
for _gpu in GPUS:
    try:
        tf.config.experimental.set_memory_growth(_gpu, True)
    except Exception:
        pass
import keras
USE_MIXED_PRECISION = len(GPUS) > 0
if USE_MIXED_PRECISION:
    keras.mixed_precision.set_global_policy("mixed_float16")
from keras.models import Model
from keras.layers import Input, Dense, Dropout, BatchNormalization, Bidirectional, GRU
from keras.optimizers import Adam
from keras.regularizers import l2
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, log_loss,
    classification_report, confusion_matrix, matthews_corrcoef,
    roc_curve, auc, precision_recall_curve,
)
np.random.seed(SEED)
tf.random.set_seed(SEED)
MODEL_NAME = "AGB-Net"
print(f"Dataset selected : {DATASET}")
print(f"GPUs visible to TensorFlow: {len(GPUS)} | "
      f"Mixed precision: {'mixed_float16' if USE_MIXED_PRECISION else 'float32 (no GPU)'}")
print(f"Python {platform.python_version()} | TensorFlow {tf.__version__} | Keras {keras.__version__} | "
      f"NumPy {np.__version__} | Pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

c:\Users\ashis\miniconda3\envs\tf\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU, compute capability 8.6
Dataset selected : cicids
GPUs visible to TensorFlow: 1 | Mixed precision: mixed_float16
Python 3.9.25 | TensorFlow 2.10.1 | Keras 2.10.0 | NumPy 1.26.4 | Pandas 2.2.3 | scikit-learn 1.5.2


## Results Persistence
All headline numbers (SBOA search results, performance tables, timing) are accumulated in the
`RESULTS` dict and written to `results/results_<dataset>.json`. Re-running with `RUN_SBOA = False`
reloads the cached search results instead of repeating the (expensive) hyperparameter search.

In [3]:
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
def _results_path(dataset=None):
    return RESULTS_DIR / f"results_{dataset or DATASET}.json"
def save_results(dataset=None):
    path = _results_path(dataset)
    with open(path, "w") as f:
        _json.dump(RESULTS, f, indent=2, default=float)
    print(f"Saved results -> {path}")
def load_results(dataset=None):
    path = _results_path(dataset)
    if path.exists():
        with open(path) as f:
            return _json.load(f)
    return {}
RESULTS = load_results(DATASET)

## 1. Load Raw Data
Each dataset has its own file layout and label scheme; loaders below return a common
`(X, y, CLASS_ORDER, NATIVE_TEST_MASK)` tuple. `NATIVE_TEST_MASK` is a boolean array marking
rows that belong to the dataset's own official test file (only meaningful for NSL-KDD's
KDDTrain+/KDDTest+ split); it is `None` for datasets that ship a single pooled file.

**Description:** One-hot encode categorical columns

In [4]:
def find_dataset_root():
    '''Locate the `dataset/` folder: check cwd, then up to two parent directories.
    Notebooks are typically run with cwd set to the notebook's own directory, but the
    shared `dataset/` folder may live a couple of levels up (e.g. next to a `Publications/`
    or project root). Checking a few parent levels lets every loader work whether the
    notebook is opened from its own folder or from somewhere above it.
    '''
    for c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (c / "dataset").exists():
            return c / "dataset"
    return Path.cwd() / "dataset"
def load_nslkdd():
    '''NSL-KDD: KDDTrain+.txt / KDDTest+.txt, manual attack-subclass -> 5-category mapping.'''
    DATA_DIR = find_dataset_root() / "nslkdd"
    for fname in ["KDDTrain+.txt", "KDDTest+.txt"]:
        if not (DATA_DIR / fname).exists():
            raise FileNotFoundError(f"Missing required dataset file: {DATA_DIR / fname}")
    col_names = ["duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
        "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
        "root_shell","su_attempted","num_root","num_file_creations","num_shells",
        "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
        "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
        "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
        "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
        "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
        "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
        "subclass","difficulty_level"]
    df_train = pd.read_csv(DATA_DIR / "KDDTrain+.txt", header=None, names=col_names).drop("difficulty_level", axis=1)
    df_test  = pd.read_csv(DATA_DIR / "KDDTest+.txt",  header=None, names=col_names).drop("difficulty_level", axis=1)
    native_test_mask = np.concatenate([np.zeros(len(df_train), dtype=bool), np.ones(len(df_test), dtype=bool)])
    combined = pd.concat([df_train, df_test], ignore_index=True)
    for c in ["protocol_type", "service", "flag"]:
        dummies = pd.get_dummies(combined[c], prefix=c, drop_first=False, dtype=np.uint8)
        combined = pd.concat([combined.drop(c, axis=1), dummies], axis=1)
    subclass = combined.pop("subclass")
    dos_set   = {"apache2","back","land","neptune","mailbomb","pod","processtable","smurf","teardrop","udpstorm","worm"}
    probe_set = {"ipsweep","mscan","nmap","portsweep","saint","satan"}
    u2r_set   = {"buffer_overflow","loadmodule","perl","ps","rootkit","sqlattack","xterm"}
    r2l_set   = {"ftp_write","guess_passwd","httptunnel","imap","multihop","named","phf","sendmail",
                 "Snmpgetattack","spy","snmpguess","warezclient","warezmaster","xlock","xsnoop"}
    def map_class(v):
        if v in dos_set:   return "DoS"
        if v in probe_set: return "Probe"
        if v in u2r_set:   return "U2R"
        if v in r2l_set:   return "R2L"
        return "Normal"
    class_order = ["DoS", "Normal", "Probe", "R2L", "U2R"]
    class_to_idx = {c: i for i, c in enumerate(class_order)}
    y = subclass.map(map_class).map(class_to_idx).astype(np.int32).values
    X = combined.values.astype(np.float32)
    return X, y, class_order, native_test_mask
def load_cicids():
    '''CICIDS2017: 8 raw per-day ISCX CSVs, concatenated and mapped to a reduced Attack Type scheme.
    The official release ships one CSV per capture day (Monday .. Friday, with Friday split
    into DDoS/PortScan/Bot captures) rather than a single pre-cleaned file. A parquet cache of
    the concatenated + labeled result is written next to the raw files so repeat runs skip the
    (slow) CSV concatenation.
    '''
    DATA_DIR = find_dataset_root() / "cicids"
    parquet_path = DATA_DIR / "cicids2017_combined.parquet"
    if parquet_path.exists():
        df = pd.read_parquet(parquet_path).drop_duplicates()
    else:
        raw_files = sorted(DATA_DIR.glob("*.pcap_ISCX.csv"))
        if not raw_files:
            raise FileNotFoundError(f"No raw CICIDS2017 *_ISCX.csv files found under {DATA_DIR}")
        frames = []
        for f in raw_files:
            d = pd.read_csv(f, low_memory=False)
            d.columns = [c.strip() for c in d.columns]
            frames.append(d)
        df = pd.concat(frames, ignore_index=True).drop_duplicates()
        df.to_parquet(parquet_path, index=False)
    def map_attack(v):
        vl = v.strip().lower()
        if vl == "benign":
            return "Normal"
        if vl.startswith("web attack"):
            return "Web Attack"
        if vl in ("ftp-patator", "ssh-patator"):
            return "Brute Force"
        if vl.startswith("dos") or vl == "heartbleed":
            return "DoS"
        if vl == "ddos":
            return "DDoS"
        if vl == "portscan":
            return "PortScan"
        if vl == "bot":
            return "Bot(s)"
        if vl == "infiltration":
            return "Infiltration"
        return "Other"
    attack_type = df["Label"].astype(str).map(map_attack)
    le = LabelEncoder()
    y = le.fit_transform(attack_type)
    class_order = list(le.classes_)
    bot_label = next((l for l in class_order if "bot" in l.lower()), None)
    if bot_label and class_order.index(bot_label) != len(class_order) - 1:
        old_order = list(le.classes_)
        class_order.remove(bot_label)
        class_order.append(bot_label)
        remap = {old_i: class_order.index(name) for old_i, name in enumerate(old_order)}
        y = np.array([remap[v] for v in y])
    drop_cols = [c for c in ["Label", "Flow ID", "Source IP", "Destination IP",
                              "Timestamp", "Src IP", "Dst IP"] if c in df.columns]
    X = df.drop(columns=drop_cols).select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan)
    return X.values.astype(np.float32), y.astype(np.int32), class_order, None
def _prep_tabular(df, label_col, drop_cols, onehot_cap=50):
    '''Shared tabular prep: drop id/leakage columns, one-hot low-cardinality categoricals,
    keep numeric features. Returns (X float32 with possible NaN, y int32, class_order).'''
    y_raw = df[label_col].astype(str).str.strip()
    drop = {label_col} | {c for c in drop_cols if c in df.columns}
    feat = df.drop(columns=list(drop))
    obj_cols = feat.select_dtypes(include="object").columns.tolist()
    keep_oh = [c for c in obj_cols if feat[c].nunique() <= onehot_cap]
    drop_hi = [c for c in obj_cols if c not in keep_oh]           
    feat = feat.drop(columns=drop_hi)
    X_num = feat.drop(columns=keep_oh).apply(pd.to_numeric, errors="coerce")
    X_cat = (pd.get_dummies(feat[keep_oh].astype(str), prefix=keep_oh, dummy_na=False)
             if keep_oh else pd.DataFrame(index=feat.index))
    X = pd.concat([X_num, X_cat], axis=1).replace([np.inf, -np.inf], np.nan)
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    return X.values.astype(np.float32), y.astype(np.int32), list(le.classes_)
def load_ciciot23():
    '''CIC-IoT-2023: pooled train/validation/test CSVs; 34 fine labels -> 8 top-level groups.'''
    D = find_dataset_root() / "CICIOT23"
    parts = [D / "train" / "train.csv", D / "validation" / "validation.csv", D / "test" / "test.csv"]
    frames = [pd.read_csv(p) for p in parts if p.exists()]
    if not frames:
        raise FileNotFoundError(f"No CICIOT23 CSVs found under {D}")
    df = pd.concat(frames, ignore_index=True)
    def grp(v):
        v = str(v)
        if v == "BenignTraffic": return "Benign"
        if v.startswith("DDoS"):  return "DDoS"
        if v.startswith("DoS"):   return "DoS"
        if v.startswith("Mirai"): return "Mirai"
        if v.startswith("Recon") or v == "VulnerabilityScan": return "Recon"
        if v in ("MITM-ArpSpoofing", "DNS_Spoofing"): return "Spoofing"
        if v == "DictionaryBruteForce": return "BruteForce"
        return "Web"   
    class_order = ["Benign", "DDoS", "DoS", "Mirai", "Recon", "Spoofing", "Web", "BruteForce"]
    c2i = {c: i for i, c in enumerate(class_order)}
    y = df["label"].map(grp).map(c2i).astype(np.int32).values
    X = (df.drop(columns=["label"]).select_dtypes(include=[np.number])
           .replace([np.inf, -np.inf], np.nan).values.astype(np.float32))
    return X, y, class_order, None
def load_edge_iiot():
    '''Edge-IIoTset (DNN CSV): 15-class Attack_type; drop identifier/hex/timestamp columns.'''
    D = find_dataset_root() / "Edge_iiot"
    df = pd.read_csv(D / "DNN-EdgeIIoT-dataset.csv", low_memory=False)
    drop_id = ["frame.time", "ip.src_host", "ip.dst_host", "arp.src.proto_ipv4",
               "arp.dst.proto_ipv4", "http.request.full_uri", "http.request.uri.query",
               "http.file_data", "http.referer", "tcp.options", "tcp.payload",
               "mqtt.msg", "dns.qry.name", "Attack_label"]
    X, y, class_order = _prep_tabular(df, "Attack_type", drop_id, onehot_cap=50)
    return X, y, class_order, None
def load_ton_iot():
    '''TON_IoT (train_test_network): 10-class `type`; drop IP / high-cardinality string identifiers.'''
    D = find_dataset_root() / "ton_iot"
    df = pd.read_csv(D / "train_test_network_full.csv")
    drop_id = ["src_ip", "dst_ip", "dns_query", "ssl_subject", "ssl_issuer",
               "http_uri", "http_user_agent", "label"]
    X, y, class_order = _prep_tabular(df, "type", drop_id, onehot_cap=50)
    return X, y, class_order, None
LOADERS = {"nslkdd": load_nslkdd, "cicids": load_cicids,
           "ciciot23": load_ciciot23, "edge_iiot": load_edge_iiot, "ton_iot": load_ton_iot}

> **Sections 2-15 are the single-dataset walkthrough.** They run only when `RUN_SINGLE_DEMO = True` (config cell) and operate on the one dataset named by `DATASET`. They are explanatory: the batch runner in section 16 recomputes every metric and every paper figure for *all* datasets independently. Leave the flag off for unattended runs.


**Description:** Optional stratified row cap for very large datasets (see MAX_SAMPLES in the config cell).

In [5]:
if RUN_SINGLE_DEMO:
    X_raw, y_all, CLASS_ORDER, NATIVE_TEST_MASK = LOADERS[DATASET]()
    _cap = MAX_SAMPLES.get(DATASET) if isinstance(MAX_SAMPLES, dict) else MAX_SAMPLES
    if _cap and len(X_raw) > _cap:
        X_raw, _, y_all, _ = train_test_split(X_raw, y_all, train_size=_cap,
                                              random_state=SEED, stratify=y_all)
        NATIVE_TEST_MASK = None
        print(f"[{DATASET}] stratified cap -> {_cap:,} rows (MAX_SAMPLES)")
    NUM_CLASSES = len(CLASS_ORDER)
    print(f"[{DATASET}] Raw features: {X_raw.shape[1]} | Samples: {X_raw.shape[0]:,} | Classes: {NUM_CLASSES}")
    for i, c in enumerate(CLASS_ORDER):
        print(f"  {c}: {(y_all==i).mean()*100:.2f}%")

## 2. Split and Shared Preprocessing
By default: 70/15/15 stratified split. If `NSL_NATIVE_SPLIT = True` and `DATASET == "nslkdd"`,
the test split instead uses the dataset's own KDDTest+ rows (which contain attack subtypes not
seen in KDDTrain+), giving a genuine distribution-shift evaluation rather than an i.i.d. random
split. Either way, imputation / augmentation / scaling / variance-filtering are fit on the
training split only, to avoid leakage.

**Description:** Median imputation (fit on train only)

In [6]:
if RUN_SINGLE_DEMO:
    if DATASET == "nslkdd" and NSL_NATIVE_SPLIT and NATIVE_TEST_MASK is not None:
        test_idx  = np.where(NATIVE_TEST_MASK)[0]
        train_idx = np.where(~NATIVE_TEST_MASK)[0]
        X_test, y_test = X_raw[test_idx], y_all[test_idx]
        X_tmp, y_tmp   = X_raw[train_idx], y_all[train_idx]
        X_train, X_val, y_train, y_val = train_test_split(
            X_tmp, y_tmp, test_size=VAL_FRAC, random_state=SEED, stratify=y_tmp
        )
        print("Using NSL-KDD's native KDDTrain+/KDDTest+ split (distribution-shift evaluation).")
    else:
        X_tmp, X_test, y_tmp, y_test = train_test_split(
            X_raw, y_all, test_size=TEST_FRAC, random_state=SEED, stratify=y_all
        )
        X_train, X_val, y_train, y_val = train_test_split(
            X_tmp, y_tmp, test_size=VAL_FRAC, random_state=SEED, stratify=y_tmp
        )
    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_train)
    X_val   = imputer.transform(X_val)
    X_test  = imputer.transform(X_test)
    def add_log(X):
        return np.hstack([X, np.log1p(np.clip(X, 0, None))])
    X_train = add_log(X_train)
    X_val   = add_log(X_val)
    X_test  = add_log(X_test)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)
    vt = VarianceThreshold(threshold=1e-4)
    X_train = vt.fit_transform(X_train)
    X_val   = vt.transform(X_val)
    X_test  = vt.transform(X_test)
    NUM_FEATURES = X_train.shape[1]
    X_train_lstm = X_train.reshape(-1, TIMESTEPS, NUM_FEATURES)
    X_val_lstm   = X_val.reshape(-1, TIMESTEPS, NUM_FEATURES)
    X_test_lstm  = X_test.reshape(-1, TIMESTEPS, NUM_FEATURES)
    print(f"Features after log1p augmentation + variance filter: {NUM_FEATURES}")
    print(f"Train: {X_train_lstm.shape} | Val: {X_val_lstm.shape} | Test: {X_test_lstm.shape}")

## 3. Feature-Correlation Graph for the GCN Front-End
A feature-feature adjacency matrix is built from absolute Pearson correlations on the training
split. `compute_corr_matrix` is computed **once**; `adjacency_from_corr` then applies a top-k
neighborhood filter and symmetric degree normalization. Separating the two lets the (expensive)
correlation matrix be reused cheaply whenever `top_k` changes -- both during the SBOA search and
when initializing the adaptive graph layer below.

In [7]:
def compute_corr_matrix(X):
    corr = np.abs(np.corrcoef(X, rowvar=False))
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    np.fill_diagonal(corr, 1.0)
    return corr
def adjacency_from_corr(corr, top_k):
    mask = np.zeros_like(corr)
    mask[np.arange(corr.shape[0])[:, None], np.argsort(corr, axis=1)[:, -top_k:]] = 1.0
    A = np.maximum(corr * mask, (corr * mask).T)
    np.fill_diagonal(A, 1.0)
    deg_inv_sqrt = np.power(A.sum(axis=1) + 1e-8, -0.5)
    return (deg_inv_sqrt[:, None] * A) * deg_inv_sqrt[None, :]
def build_feature_adjacency(X, top_k=8):
    return adjacency_from_corr(compute_corr_matrix(X), top_k)

In [8]:
if RUN_SINGLE_DEMO:
    CORR_TRAIN = compute_corr_matrix(X_train)
    A_norm = adjacency_from_corr(CORR_TRAIN, TOP_K)
    print(f"Adjacency: {A_norm.shape}")

## 4. Adaptive Graph Convolution Layer (Phase 3A)
`AdaptiveGCN` generalizes the original fixed-adjacency graph convolution:

- **`adaptive=True`** (default): the adjacency is a **trainable weight**, warm-started from the
  Pearson-correlation graph above (`init_adjacency`) so training starts from an interpretable,
  data-driven prior. On every forward pass the raw weight is projected back onto a valid
  normalized-graph manifold: symmetrize `(A + Aᵀ)/2`, `ReLU` for non-negative edge weights,
  top-k sparsify, then symmetric degree normalization. This lets the network reshape which
  features it treats as connected while keeping the graph well-formed at every step.
- **`adaptive=False`**: the Pearson graph is used as a fixed constant, exactly recovering the
  original `SimpleGCN` behaviour. Toggling `ADAPTIVE_GRAPH` therefore gives a built-in
  fixed-vs-learned-graph ablation at no extra implementation cost.

**Description:** enforce symmetry

In [9]:
class AdaptiveGCN(keras.layers.Layer):
    def __init__(self, units, init_adjacency, top_k=8, adaptive=True,
                 activation="relu", l2_reg=1e-3, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.init_adjacency = np.asarray(init_adjacency, dtype=np.float32)
        self.top_k = top_k
        self.adaptive = adaptive
        self.activation = keras.activations.get(activation)
        self.l2_reg = l2_reg
    def build(self, input_shape):
        in_dim = int(input_shape[-1])
        if self.adaptive:
            self.adjacency_raw = self.add_weight(
                name="adjacency_raw", shape=self.init_adjacency.shape,
                initializer=keras.initializers.Constant(self.init_adjacency),
                trainable=True,
            )
        else:
            self.adjacency_raw = tf.constant(self.init_adjacency, dtype=tf.float32)
        self.kernel = self.add_weight(name="kernel", shape=(in_dim, self.units),
                                      initializer="glorot_uniform",
                                      regularizer=l2(self.l2_reg), trainable=True)
        self.bias = self.add_weight(name="bias", shape=(self.units,), initializer="zeros", trainable=True)
    def _normalized_adjacency(self):
        A = tf.cast(self.adjacency_raw, tf.float32)
        if self.adaptive:
            A = 0.5 * (A + tf.transpose(A))            
            A = tf.nn.relu(A)                            
            k = min(self.top_k, A.shape[-1])
            top_vals, _ = tf.math.top_k(A, k=k)
            thresh = top_vals[:, -1:]                     
            mask = tf.cast(A >= thresh, A.dtype)
            A = A * mask
            A = tf.maximum(A, tf.transpose(A))            
        deg = tf.reduce_sum(A, axis=-1) + 1e-8
        deg_inv_sqrt = tf.pow(deg, -0.5)
        return A * deg_inv_sqrt[:, None] * deg_inv_sqrt[None, :]
    def call(self, inputs):
        adj = tf.cast(self._normalized_adjacency(), inputs.dtype)
        x = tf.matmul(inputs, adj)
        x = tf.matmul(x, tf.cast(self.kernel, x.dtype)) + tf.cast(self.bias, x.dtype)
        out = self.activation(x) if self.activation is not None else x
        return out

## 4b. Adjacency Comparison (RQ2 support)

`export_adjacency_comparison` extracts the learned adjacency from a trained model's
`AdaptiveGCN` layer and compares it against the initial Pearson-correlation adjacency it
was warm-started from -- a side-by-side heatmap plus edge-change statistics, so the
adaptive-graph mechanism is empirically visible rather than only asserted in prose.

In [10]:
def neighbor_mask_from_corr(corr, top_k):
    """Binary top-k neighbourhood mask (symmetrized), the same candidate-edge set used by
    adjacency_from_corr -- shared with the GAT-style baseline below so both graphs start
    from an identical neighbourhood structure."""
    mask = np.zeros_like(corr)
    mask[np.arange(corr.shape[0])[:, None], np.argsort(corr, axis=1)[:, -top_k:]] = 1.0
    mask = np.maximum(mask, mask.T)
    np.fill_diagonal(mask, 1.0)
    return mask.astype(np.float32)
def get_learned_adjacency(model):
    """Find the AdaptiveGCN layer inside a trained model and return its final normalized
    adjacency (post symmetrize / ReLU / top-k / degree-normalize) as a numpy array."""
    for layer in model.layers:
        if isinstance(layer, AdaptiveGCN):
            return layer._normalized_adjacency().numpy()
    raise ValueError("No AdaptiveGCN layer found in model")
def export_adjacency_comparison(model, corr_train, top_k, dataset):
    """Compares the learned (post-training) adjacency against the initial Pearson-correlation
    adjacency it was warm-started from: saves a side-by-side heatmap plus summary statistics
    (edges added/removed/kept, Frobenius distance, edge-weight correlation) to
    results/adjacency_<dataset>.json. This makes the paper's central "adaptive graph" claim
    empirically visible rather than only asserted in prose."""
    A_init = adjacency_from_corr(corr_train, top_k)
    A_learned = get_learned_adjacency(model)
    n = A_init.shape[0]
    off_diag = ~np.eye(n, dtype=bool)
    edges_init = (A_init > 1e-6) & off_diag
    edges_learned = (A_learned > 1e-6) & off_diag
    added = int(np.sum(edges_learned & ~edges_init))
    removed = int(np.sum(edges_init & ~edges_learned))
    kept = int(np.sum(edges_init & edges_learned))
    frob_dist = float(np.linalg.norm(A_learned - A_init))
    both_edges = (edges_init | edges_learned)
    edge_corr = (float(np.corrcoef(A_init[both_edges], A_learned[both_edges])[0, 1])
                 if both_edges.sum() > 1 else float("nan"))
    stats = {
        "edges_kept": kept, "edges_added": added, "edges_removed": removed,
        "frobenius_distance": frob_dist, "edge_weight_correlation": edge_corr,
        "num_features": int(n), "top_k": int(top_k),
    }
    print(f"[{dataset}] Adjacency change vs. initial correlation graph: "
          f"kept={kept}, added={added}, removed={removed}, "
          f"Frobenius dist={frob_dist:.3f}, edge-weight corr={edge_corr:.3f}")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    vmax = max(A_init.max(), A_learned.max())
    sns.heatmap(A_init, ax=axes[0], cmap="viridis", vmin=0, vmax=vmax, cbar=True, square=True)
    axes[0].set_title(f"Initial correlation adjacency ({dataset})", fontweight="bold")
    sns.heatmap(A_learned, ax=axes[1], cmap="viridis", vmin=0, vmax=vmax, cbar=True, square=True)
    axes[1].set_title(f"Learned adjacency after training ({dataset})", fontweight="bold")
    diff = A_learned - A_init
    dmax = float(np.abs(diff).max()) + 1e-9
    sns.heatmap(diff, ax=axes[2], cmap="coolwarm", vmin=-dmax, vmax=dmax, cbar=True, square=True)
    axes[2].set_title("Learned minus initial (difference)", fontweight="bold")
    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.savefig(f"adjacency_comparison_{dataset}.png", dpi=150, bbox_inches="tight")
    plt.show()
    with open(RESULTS_DIR / f"adjacency_{dataset}.json", "w") as f:
        _json.dump(stats, f, indent=2, default=float)
    return stats

## 5. GCN-BiGRU Architecture
`AdaptiveGCN` front-end -> two stacked Bidirectional GRU blocks -> dense head -> softmax
classification. GRU widths, dropout, regularization strength, adjacency `top_k`, and the
optimizer are all exposed as arguments so the SBOA search (Phase 1) can vary them; defaults
match the original fixed recipe.

**Description:** dtype=float32 ensures numerically stable softmax under mixed_float16

In [11]:
def build_gcn_bigru(num_features, num_classes, adjacency, timesteps=1, dropout=0.3, reg=1e-3,
                     bigru1=128, bigru2=64, top_k=None, adaptive=None):
    if top_k is None:
        top_k = TOP_K
    if adaptive is None:
        adaptive = ADAPTIVE_GRAPH
    inp = Input(shape=(timesteps, num_features))
    x = AdaptiveGCN(num_features, init_adjacency=adjacency, top_k=top_k,
                    adaptive=adaptive, activation="relu", l2_reg=reg)(inp)
    for units, ret_seq in [(bigru1, True), (bigru2, False)]:
        x = Bidirectional(GRU(units, return_sequences=ret_seq,
                              kernel_regularizer=l2(reg),
                              recurrent_regularizer=l2(reg)))(x)
        x = BatchNormalization()(x)
        x = Dropout(dropout)(x)
    x = Dense(64, activation="relu", kernel_regularizer=l2(reg))(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout / 2)(x)
    x = Dense(32, activation="relu")(x)
    x = Dropout(dropout / 2)(x)
    out = Dense(num_classes, activation="softmax", dtype="float32", name="anomaly_output")(x)
    return Model(inp, out, name=f"GCN_BiGRU_{DATASET}")
OPTIMIZERS = {
    "adam": Adam,
    "rmsprop": keras.optimizers.RMSprop,
    "sgd": keras.optimizers.SGD,
    "adadelta": keras.optimizers.Adadelta,
}

## 5b. Baseline Architectures (RQ1)

`build_bigru_only` (no graph front-end) and `build_gat_bigru` (single-head GAT-style
attention over the same top-k feature-correlation neighbourhood) are re-implemented
here so they can be trained and evaluated on the exact same preprocessing, splits, class
weights, and hardware as AGB-Net, giving a controlled comparison rather than citing
accuracy numbers from other papers.

**Description:** (F, F) binary candidate edges

In [12]:
def build_bigru_only(num_features, num_classes, timesteps=1, dropout=0.3, reg=1e-3,
                      bigru1=128, bigru2=64):
    """Baseline: stacked BiGRU directly on raw (preprocessed) features -- no graph front-end
    at all. Re-implemented on the same preprocessing/splits/hardware as AGB-Net so the
    comparison is controlled, rather than citing another paper's number."""
    inp = Input(shape=(timesteps, num_features))
    x = inp
    for units, ret_seq in [(bigru1, True), (bigru2, False)]:
        x = Bidirectional(GRU(units, return_sequences=ret_seq,
                              kernel_regularizer=l2(reg), recurrent_regularizer=l2(reg)))(x)
        x = BatchNormalization()(x)
        x = Dropout(dropout)(x)
    x = Dense(64, activation="relu", kernel_regularizer=l2(reg))(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout / 2)(x)
    x = Dense(32, activation="relu")(x)
    x = Dropout(dropout / 2)(x)
    out = Dense(num_classes, activation="softmax", dtype="float32", name="anomaly_output")(x)
    return Model(inp, out, name=f"BiGRU_only_{DATASET}")
class SimpleGAT(keras.layers.Layer):
    """Single-head GAT-style attention over the same top-k feature-correlation neighbourhood
    mask used by AdaptiveGCN. Each "node" here is a single scalar feature value (not a feature
    vector, since the graph is over features rather than flow records), so attention logits are
    computed directly from the two endpoint scalars following the additive-attention form of
    Velickovic et al. (GAT): e_ij = LeakyReLU(a_i * x_i + a_j * x_j), softmax-normalized over the
    allowed neighbourhood per row. This gives an independent, literature-comparable attention
    baseline distinct from AdaptiveGCN's (fixed or learned) correlation-weighted adjacency."""
    def __init__(self, units, neighbor_mask, activation="relu", l2_reg=1e-3, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.mask = tf.constant(neighbor_mask, dtype=tf.float32)  
        self.activation = keras.activations.get(activation)
        self.l2_reg = l2_reg
    def build(self, input_shape):
        F = int(input_shape[-1])
        self.a_i = self.add_weight(name="a_i", shape=(1,), initializer="glorot_uniform", trainable=True)
        self.a_j = self.add_weight(name="a_j", shape=(1,), initializer="glorot_uniform", trainable=True)
        self.kernel = self.add_weight(name="kernel", shape=(F, self.units),
                                      initializer="glorot_uniform", regularizer=l2(self.l2_reg), trainable=True)
        self.bias = self.add_weight(name="bias", shape=(self.units,), initializer="zeros", trainable=True)
    def call(self, inputs):
        x = tf.squeeze(inputs, axis=1)                          
        xi = tf.expand_dims(x, 2)                                 
        xj = tf.expand_dims(x, 1)                                 
        logits = self.a_i * xi + self.a_j * xj                    
        logits = tf.nn.leaky_relu(logits, alpha=0.2)
        neg_inf = tf.fill(tf.shape(logits), tf.cast(-1e9, logits.dtype))
        mask_b = self.mask[None, :, :] > 0
        masked = tf.where(mask_b, logits, neg_inf)
        alpha = tf.nn.softmax(masked, axis=-1)
        alpha = tf.where(mask_b, alpha, tf.zeros_like(alpha))
        agg = tf.matmul(alpha, tf.expand_dims(x, 2))[:, :, 0]      
        out = tf.matmul(agg, self.kernel) + self.bias
        out = self.activation(out) if self.activation is not None else out
        return tf.expand_dims(out, axis=1)
def build_gat_bigru(num_features, num_classes, neighbor_mask, timesteps=1, dropout=0.3, reg=1e-3,
                     bigru1=128, bigru2=64):
    """Baseline: GAT-style attention front-end (see SimpleGAT) + the same stacked-BiGRU
    backbone as AGB-Net, re-implemented on identical preprocessing/splits/hardware."""
    inp = Input(shape=(timesteps, num_features))
    x = SimpleGAT(num_features, neighbor_mask, activation="relu", l2_reg=reg)(inp)
    for units, ret_seq in [(bigru1, True), (bigru2, False)]:
        x = Bidirectional(GRU(units, return_sequences=ret_seq,
                              kernel_regularizer=l2(reg), recurrent_regularizer=l2(reg)))(x)
        x = BatchNormalization()(x)
        x = Dropout(dropout)(x)
    x = Dense(64, activation="relu", kernel_regularizer=l2(reg))(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout / 2)(x)
    x = Dense(32, activation="relu")(x)
    x = Dropout(dropout / 2)(x)
    out = Dense(num_classes, activation="softmax", dtype="float32", name="anomaly_output")(x)
    return Model(inp, out, name=f"GAT_BiGRU_{DATASET}")

## 6. Class Weights
Sqrt-inverse-frequency class weights soften the imbalance without the extreme ratios of raw
inverse-frequency weighting.

**Description:** normalize so average weight = 1

In [13]:
if RUN_SINGLE_DEMO:
    freq = np.bincount(y_train, minlength=NUM_CLASSES).astype(float) / len(y_train)
    cw = 1.0 / np.sqrt(freq + 1e-8)
    cw = cw / cw.mean()          
    cw = np.clip(cw, 0.5, 15.0)  
    CLASS_WEIGHTS = dict(enumerate(cw))
    print(f"Class weights: {dict(zip(CLASS_ORDER, np.round(cw, 3)))}")

## 7. Secretary Bird Optimization Algorithm (SBOA) -- Hyperparameter Search (Phase 1)
Implements the SBOA structure described in the paper's Algorithm 1 / Eqs. (6)-(12): each
candidate is a real vector in `[0,1]^D` (decoded per-dimension into an actual hyperparameter
value), and every iteration updates every population member through two sequential phases:

1. **Hunting strategy (exploration)**, gated by the iteration counter `t` against the total
   budget `T`:
   - *Stage 1* (`t < T/3`, searching for prey): a random-walk step toward two random peers.
   - *Stage 2* (`T/3 <= t < 2T/3`, consuming prey): a Brownian-motion step around the current best.
   - *Stage 3* (`t >= 2T/3`, attacking prey): a Lévy-flight step guided toward the current best.
2. **Escaping strategy (exploitation / anti-predation)**, one of two moves chosen with equal
   probability:
   - **C1 -- camouflage**: a small perturbation around the current position that shrinks as
     `t -> T`.
   - **C2 -- flight**: a jump around the current best, scaled by a shrinking random envelope.

Each candidate move is accepted only if it improves that individual's fitness (greedy selection),
matching the elitist structure of Algorithm 1. Fitness is validation accuracy of a GCN-BiGRU
model trained for a **reduced budget** (`SBOA_EPOCHS` epochs) on a **subsample**
(`SBOA_SAMPLE_FRAC` of the training split), keeping the search tractable on a GPU.

**Description:** --- Search space -----------------------------------------------------------

In [14]:
H = {
    "bigru1":    {"type": "int",   "low": 32,   "high": 256},
    "bigru2":    {"type": "int",   "low": 16,   "high": 128},
    "lr":        {"type": "log",   "low": 1e-4, "high": 5e-3},
    "dropout":   {"type": "float", "low": 0.1,  "high": 0.5},
    "optimizer": {"type": "cat",   "choices": ["adam", "rmsprop", "sgd", "adadelta"]},
    "top_k":     {"type": "int",   "low": 4,    "high": 16},
}
DIM = len(H)
def decode_theta(v):
    v = np.clip(v, 0.0, 1.0)
    theta = {}
    for j, (name, spec) in enumerate(H.items()):
        vi = v[j]
        if spec["type"] == "int":
            theta[name] = int(round(spec["low"] + vi * (spec["high"] - spec["low"])))
        elif spec["type"] == "log":
            lo, hi = np.log(spec["low"]), np.log(spec["high"])
            theta[name] = float(np.exp(lo + vi * (hi - lo)))
        elif spec["type"] == "float":
            theta[name] = float(spec["low"] + vi * (spec["high"] - spec["low"]))
        elif spec["type"] == "cat":
            idx = min(int(vi * len(spec["choices"])), len(spec["choices"]) - 1)
            theta[name] = spec["choices"][idx]
    return theta
def levy_flight(dim, beta=1.5):
    '''Mantegna's algorithm for Levy-distributed step sizes, clipped to [-1, 1].'''
    sigma_u = (sp_gamma(1 + beta) * np.sin(np.pi * beta / 2) /
               (sp_gamma((1 + beta) / 2) * beta * 2 ** ((beta - 1) / 2))) ** (1 / beta)
    u = np.random.normal(0, sigma_u, size=dim)
    v = np.random.normal(0, 1, size=dim)
    step = u / (np.power(np.abs(v), 1 / beta) + 1e-12)
    return np.clip(step, -1.0, 1.0)

In [15]:
def sboa_fitness(theta_vec, X_tr, y_tr, X_va, y_va, corr_matrix, num_classes, num_features, timesteps):
    theta = decode_theta(theta_vec)
    A = adjacency_from_corr(corr_matrix, theta["top_k"])
    tf.keras.backend.clear_session()
    m = build_gcn_bigru(num_features, num_classes, A, timesteps,
                         dropout=theta["dropout"], reg=REG,
                         bigru1=theta["bigru1"], bigru2=theta["bigru2"],
                         top_k=theta["top_k"], adaptive=ADAPTIVE_GRAPH)
    opt = OPTIMIZERS[theta["optimizer"]](learning_rate=theta["lr"], clipnorm=1.0)
    m.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    es = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, mode="min", verbose=0)
    hist = m.fit(X_tr, y_tr, validation_data=(X_va, y_va),
                 epochs=SBOA_EPOCHS, batch_size=BATCH, callbacks=[es], verbose=0)
    val_acc = float(max(hist.history.get("val_accuracy", [0.0])))
    del m
    return val_acc, theta

**Description:** ---- Phase 1: Hunting strategy (exploration) ----

In [16]:
def run_sboa(fitness_fn, dim, pop_size, iters, seed=SEED):
    rng = np.random.default_rng(seed)
    X = rng.random((pop_size, dim))
    evals = [fitness_fn(X[i]) for i in range(pop_size)]
    fitness = np.array([e[0] for e in evals])
    thetas  = [e[1] for e in evals]
    best_idx = int(np.argmax(fitness))
    best_x, best_f = X[best_idx].copy(), float(fitness[best_idx])
    convergence = [best_f]
    for t in range(1, iters + 1):
        for i in range(pop_size):
            r1, r2 = rng.integers(0, pop_size, size=2)
            if t < iters / 3:
                R1 = rng.random(dim)
                cand = X[i] + (X[r1] - X[r2]) * R1
            elif t < 2 * iters / 3:
                RB = rng.normal(size=dim)
                cand = best_x + np.exp((t / iters) ** 4) * (RB - 0.5) * (best_x - X[i])
            else:
                cand = best_x + levy_flight(dim) * (best_x - X[i]) * (1 - t / iters)
            cand = np.clip(cand, 0.0, 1.0)
            cand_f, cand_theta = fitness_fn(cand)
            if cand_f > fitness[i]:
                X[i], fitness[i], thetas[i] = cand, cand_f, cand_theta
            if rng.random() < 0.5:
                K1 = 2 * (1 - t / iters)
                cand2 = X[i] + K1 * (rng.random(dim) * 2 - 1) * X[i]
            else:
                K2 = 0.5 * (1 - t / iters) ** 2
                cand2 = best_x + K2 * (rng.random(dim) * 2 - 1)
            cand2 = np.clip(cand2, 0.0, 1.0)
            cand2_f, cand2_theta = fitness_fn(cand2)
            if cand2_f > fitness[i]:
                X[i], fitness[i], thetas[i] = cand2, cand2_f, cand2_theta
        gen_best = int(np.argmax(fitness))
        if fitness[gen_best] > best_f:
            best_f, best_x = float(fitness[gen_best]), X[gen_best].copy()
        convergence.append(best_f)
        print(f"[SBOA] iter {t:2d}/{iters} | best val_accuracy so far: {best_f*100:.2f}%")
    best_theta = decode_theta(best_x)
    return best_x, best_theta, best_f, convergence

**Description:** Hyperparameters found by real SBOA searches, kept separately per dataset so that

In [17]:
DEFAULT_THETA_BY_DATASET = {
    "cicids": {"bigru1": 32, "bigru2": 84, "lr": 0.00024017660564659387,
               "dropout": 0.10044824369377355, "optimizer": "adam", "top_k": 6},
    "nslkdd": {"bigru1": 46, "bigru2": 81, "lr": 0.0008633556905251802,
               "dropout": 0.45187122043261396, "optimizer": "rmsprop", "top_k": 4},
    "ciciot23":  {"bigru1": 128, "bigru2": 64, "lr": LR, "dropout": DROPOUT,
                  "optimizer": "adam", "top_k": TOP_K},
    "edge_iiot": {"bigru1": 128, "bigru2": 64, "lr": LR, "dropout": DROPOUT,
                  "optimizer": "adam", "top_k": TOP_K},
    "ton_iot":   {"bigru1": 128, "bigru2": 64, "lr": LR, "dropout": DROPOUT,
                  "optimizer": "adam", "top_k": TOP_K},
}
DEFAULT_THETA = DEFAULT_THETA_BY_DATASET[DATASET]

In [18]:
if RUN_SINGLE_DEMO:
    if RUN_SBOA:
        n_sub = max(int(len(X_train) * SBOA_SAMPLE_FRAC), BATCH * 4)
        rng_sel = np.random.default_rng(SEED)
        sub_idx = rng_sel.choice(len(X_train), size=min(n_sub, len(X_train)), replace=False)
        X_sboa, y_sboa = X_train[sub_idx], y_train[sub_idx]
        X_sb_tr, X_sb_va, y_sb_tr, y_sb_va = train_test_split(
            X_sboa, y_sboa, test_size=0.2, random_state=SEED, stratify=y_sboa
        )
        X_sb_tr_lstm = X_sb_tr.reshape(-1, TIMESTEPS, NUM_FEATURES)
        X_sb_va_lstm = X_sb_va.reshape(-1, TIMESTEPS, NUM_FEATURES)
        CORR_SBOA = compute_corr_matrix(X_sb_tr)
        def _fitness(v):
            return sboa_fitness(v, X_sb_tr_lstm, y_sb_tr, X_sb_va_lstm, y_sb_va,
                                 CORR_SBOA, NUM_CLASSES, NUM_FEATURES, TIMESTEPS)
        print(f"[SBOA] Searching on a {len(X_sboa):,}-sample subsample "
              f"({SBOA_SAMPLE_FRAC*100:.0f}% of training data), P={SBOA_POP}, T={SBOA_ITERS}, "
              f"{SBOA_EPOCHS} epochs/candidate.")
        t_sboa0 = time.perf_counter()
        best_x, theta_star, best_f, convergence = run_sboa(
            _fitness, dim=DIM, pop_size=SBOA_POP, iters=SBOA_ITERS, seed=SEED
        )
        sboa_time = time.perf_counter() - t_sboa0
        print(f"\n[SBOA] Best hyperparameters found (val_accuracy={best_f*100:.2f}%, "
              f"search time={sboa_time:.1f}s):")
        for k, v in theta_star.items():
            print(f"  {k}: {v}")
        RESULTS["sboa"] = {
            "theta_star": theta_star, "best_val_accuracy": best_f, "convergence": convergence,
            "search_time_sec": sboa_time, "pop_size": SBOA_POP, "iters": SBOA_ITERS,
            "sample_frac": SBOA_SAMPLE_FRAC, "epochs_per_candidate": SBOA_EPOCHS,
        }
        save_results()
    else:
        cached = RESULTS.get("sboa", {})
        theta_star = cached.get("theta_star", DEFAULT_THETA)
        convergence = cached.get("convergence", [])
        print("RUN_SBOA=False -> using cached/default theta*:")
        for k, v in theta_star.items():
            print(f"  {k}: {v}")

## 7b. SBOA Convergence and Selected Hyperparameters

In [19]:
if RUN_SINGLE_DEMO:
    if convergence:
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(np.arange(1, len(convergence) + 1), np.array(convergence) * 100,
                marker="o", ms=4, color="darkorange", lw=2)
        ax.set_xlabel("SBOA Iteration"); ax.set_ylabel("Best Validation Accuracy (%)")
        ax.set_title(f"SBOA Convergence -- {DATASET}", fontsize=13, fontweight="bold")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(f"sboa_convergence_{DATASET}.png", dpi=150, bbox_inches="tight")
        plt.show()
    else:
        print("No convergence history available (RUN_SBOA=False and no cached history).")

In [20]:
if RUN_SINGLE_DEMO:
    theta_df = pd.DataFrame(
        [{"Hyperparameter": k, "Value": v} for k, v in theta_star.items()]
    ).set_index("Hyperparameter")
    print(f"SBOA-selected hyperparameters ({DATASET}):")
    display(theta_df)
    lines = []
    lines.append(r"\begin{table}[htbp]")
    lines.append(r"\caption{SBOA-selected hyperparameters for the GCN-BiGRU model (" + DATASET + r")}\label{tab:sboa_theta_" + DATASET + "}")
    lines.append(r"\centering")
    lines.append(r"\begin{tabular}{@{}ll@{}}")
    lines.append(r"\toprule")
    lines.append(r"Hyperparameter & Value \\")
    lines.append(r"\midrule")
    for k, v in theta_star.items():
        vs = f"{v:.5g}" if isinstance(v, float) else str(v)
        lines.append(f"{k} & {vs} " + r"\\")
    lines.append(r"\botrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")
    print("\n--- LaTeX ---")
    print("\n".join(lines))

## 8. Final Model Training at Full Budget
The final model is built with the SBOA-selected hyperparameters `theta_star` and trained at the
full epoch budget (`EPOCHS`) on the full training split (`FINAL_FULL_DATA=True`), or on the same
subsample used for the SBOA search when `FINAL_FULL_DATA=False` (useful for a quick end-to-end
smoke test).

In [21]:
if RUN_SINGLE_DEMO:
    FINAL_TOP_K = theta_star["top_k"]
    A_final = adjacency_from_corr(CORR_TRAIN, FINAL_TOP_K)
    tf.keras.backend.clear_session()
    np.random.seed(SEED)
    tf.random.set_seed(SEED)
    model = build_gcn_bigru(NUM_FEATURES, NUM_CLASSES, A_final, TIMESTEPS,
                             dropout=theta_star["dropout"], reg=REG,
                             bigru1=theta_star["bigru1"], bigru2=theta_star["bigru2"],
                             top_k=FINAL_TOP_K, adaptive=ADAPTIVE_GRAPH)
    opt = OPTIMIZERS[theta_star["optimizer"]](learning_rate=theta_star["lr"], clipnorm=1.0)
    model.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.summary()

**Description:** restore_best_weights=True -> model already carries the best checkpoint

In [22]:
if RUN_SINGLE_DEMO:
    if FINAL_FULL_DATA or not RUN_SBOA:
        X_fit, y_fit = X_train_lstm, y_train
        X_fit_val, y_fit_val = X_val_lstm, y_val
    else:
        X_fit, y_fit = X_sb_tr_lstm, y_sb_tr
        X_fit_val, y_fit_val = X_sb_va_lstm, y_sb_va
    AUTOTUNE = tf.data.AUTOTUNE
    def make_dataset(X, y, shuffle=False):
        with tf.device("/CPU:0"):
            ds = tf.data.Dataset.from_tensor_slices((X, y))
        if shuffle:
            ds = ds.shuffle(buffer_size=20_000, seed=SEED)
        return ds.batch(BATCH).prefetch(AUTOTUNE)
    train_ds = make_dataset(X_fit, y_fit, shuffle=True)
    val_ds   = make_dataset(X_fit_val, y_fit_val)
    ckpt_path = f"best_model_{DATASET}.weights.h5"
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, mode="min", verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-7, verbose=1),
        ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True,
                        mode="min", save_weights_only=True, verbose=0),
    ]
    t_train0 = time.perf_counter()
    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=EPOCHS, callbacks=callbacks,
        class_weight=CLASS_WEIGHTS, verbose=1,
    )
    train_time_sec = time.perf_counter() - t_train0
    best_model = model

## 9. Training History

**Description:** Smooth validation curves with a moving average; raw curves are noisy epoch-to-epoch.

In [23]:
if RUN_SINGLE_DEMO:
    acc      = np.array(history.history.get("accuracy", []), dtype=float)
    val_acc  = np.array(history.history.get("val_accuracy", []), dtype=float)
    loss     = np.array(history.history.get("loss", []), dtype=float)
    val_loss = np.array(history.history.get("val_loss", []), dtype=float)
    epochs_range = np.arange(1, len(acc) + 1)
    W = max(3, len(acc) // 10)
    def smooth(x):
        return uniform_filter1d(x, size=W, mode="nearest") if len(x) else x
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    ax1.plot(epochs_range, acc, color="steelblue", lw=2, label="Train Accuracy")
    ax1.plot(epochs_range, smooth(val_acc), color="orange", lw=2, label="Val Accuracy")
    ax1.set_title("Accuracy", fontsize=13, fontweight="bold")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy")
    ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.plot(epochs_range, loss, color="steelblue", lw=2, label="Train Loss")
    ax2.plot(epochs_range, smooth(val_loss), color="orange", lw=2, label="Val Loss")
    ax2.set_title("Loss", fontsize=13, fontweight="bold")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
    ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.suptitle(f"{MODEL_NAME} Training History ({DATASET})", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(f"training_history_{DATASET}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 10. Performance Table (Train / Val / Test)

In [24]:
if RUN_SINGLE_DEMO:
    def evaluate_split(Xs, ys):
        probs = best_model.predict(Xs, batch_size=BATCH, verbose=0)
        preds = np.argmax(probs, axis=1)
        return (
            accuracy_score(ys, preds) * 100,
            precision_score(ys, preds, average="weighted", zero_division=0) * 100,
            recall_score(ys, preds, average="weighted", zero_division=0) * 100,
            f1_score(ys, preds, average="weighted", zero_division=0) * 100,
            log_loss(ys, probs, labels=np.arange(NUM_CLASSES)),
        )
    train_m = evaluate_split(X_train_lstm, y_train)
    val_m   = evaluate_split(X_val_lstm, y_val)
    test_m  = evaluate_split(X_test_lstm, y_test)
    rows_perf = [("Training", train_m), ("Validation", val_m), ("Testing", test_m)]
    perf_df = pd.DataFrame([
        {"Dataset Split": split, "Accuracy (%)": round(m[0], 2), "Precision (%)": round(m[1], 2),
         "Recall (%)": round(m[2], 2), "F1-score (%)": round(m[3], 2), "Loss": round(m[4], 4)}
        for split, m in rows_perf
    ]).set_index("Dataset Split")
    print(f"Performance on Training, Validation and Testing datasets ({MODEL_NAME}, {DATASET}):")
    display(perf_df)
    RESULTS["performance"] = {
        split: {"accuracy": m[0], "precision": m[1], "recall": m[2], "f1": m[3], "loss": m[4]}
        for split, m in rows_perf
    }
    RESULTS["config"] = {
        "adaptive_graph": ADAPTIVE_GRAPH, "theta_star": theta_star,
        "final_full_data": FINAL_FULL_DATA, "train_time_sec": train_time_sec,
    }
    save_results()
    lines = []
    lines.append(r"\begin{table}[htbp]")
    lines.append(r"\caption{Performance analysis of the proposed AGB-Net model on training, validation, and testing datasets (" + DATASET + r")}\label{tab:performance_" + DATASET + "}")
    lines.append(r"\centering")
    lines.append(r"\begin{tabular}{@{}lccccc@{}}")
    lines.append(r"\toprule")
    lines.append(r"Dataset Split & Accuracy (\%) & Precision (\%) & Recall (\%) & F1-score (\%) & Loss \\")
    lines.append(r"\midrule")
    for split, m in rows_perf:
        lines.append(f"{split} & {m[0]:.2f} & {m[1]:.2f} & {m[2]:.2f} & {m[3]:.2f} & {m[4]:.4f} " + r"\\")
    lines.append(r"\botrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")
    print("\n--- LaTeX ---")
    print("\n".join(lines))

## 11. Confusion Matrix -- True vs Predicted Attack Labels

In [25]:
if RUN_SINGLE_DEMO:
    y_prob_test = best_model.predict(X_test_lstm, batch_size=BATCH, verbose=0)
    pred        = np.argmax(y_prob_test, axis=1)
    y_eval      = y_test
    print("Test-set classification report:")
    print(classification_report(y_eval, pred, target_names=CLASS_ORDER, zero_division=0))
    y_true_lbl = np.array([CLASS_ORDER[i] for i in y_eval])
    y_pred_lbl = np.array([CLASS_ORDER[i] for i in pred])
    labels = CLASS_ORDER.copy()
    cm_counts = confusion_matrix(y_true_lbl, y_pred_lbl, labels=labels)
    cm_pct = cm_counts.astype(float) / (cm_counts.sum(axis=1, keepdims=True) + 1e-9) * 100
    annot = np.array([[f"{v:.1f}%" for v in row] for row in cm_pct])
    fig, ax = plt.subplots(figsize=(max(10, len(labels) * 1.3), max(7, len(labels) * 1.0)))
    sns.heatmap(
        cm_pct, annot=annot, fmt="", cmap="Blues", vmin=0, vmax=100,
        cbar_kws={"label": "% of True Class"},
        xticklabels=labels, yticklabels=labels,
        linewidths=0.5, linecolor="white", ax=ax,
    )
    ax.set_title(f"Confusion Matrix - {MODEL_NAME} ({DATASET})", fontsize=13, fontweight="bold")
    ax.set_ylabel("True Label", fontsize=11)
    ax.set_xlabel("Predicted Label", fontsize=11)
    ax.tick_params(axis="x", rotation=30)
    ax.tick_params(axis="y", rotation=0)
    plt.tight_layout()
    plt.savefig(f"confusion_matrix_{DATASET}.png", dpi=150, bbox_inches="tight")
    plt.show()
    det = pd.DataFrame({
        "True Label": labels,
        "Total Samples": cm_counts.sum(axis=1),
        "Correctly Predicted": cm_counts.diagonal(),
        "Detection Rate (%)": cm_pct.diagonal().round(2),
    }).set_index("True Label")
    print("Per-class Detection Rate:")
    display(det)

## 12. Class-wise Additional Performance Indicators
FPR, FNR, FDR, FOR, TNR, MCC, and NPV per attack class.

In [26]:
if RUN_SINGLE_DEMO:
    rows = []
    for i, label in enumerate(labels):
        y_tb = (np.array(y_true_lbl) == label).astype(int)
        y_pb = (np.array(y_pred_lbl) == label).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_tb, y_pb, labels=[0, 1]).ravel()
        fpr = fp / (fp + tn) * 100 if (fp + tn) > 0 else 0.0
        fnr = fn / (fn + tp) * 100 if (fn + tp) > 0 else 0.0
        fdr = fp / (fp + tp) * 100 if (fp + tp) > 0 else 0.0
        for_ = fn / (fn + tn) * 100 if (fn + tn) > 0 else 0.0
        tnr = tn / (tn + fp) * 100 if (tn + fp) > 0 else 0.0
        npv = tn / (tn + fn) * 100 if (tn + fn) > 0 else 0.0
        mcc = matthews_corrcoef(y_tb, y_pb) if len(np.unique(y_tb)) > 1 and len(np.unique(y_pb)) > 1 else 0.0
        rows.append({
            "Class": f"Class {i} ({label})",
            "FPR (%)": round(fpr, 4), "FNR (%)": round(fnr, 4),
            "FDR (%)": round(fdr, 4), "FOR (%)": round(for_, 4),
            "TNR (%)": round(tnr, 4), "MCC": round(mcc, 4), "NPV (%)": round(npv, 4),
        })
    df_adv = pd.DataFrame(rows).set_index("Class")
    print(f"Class-wise Additional Performance Indicators ({MODEL_NAME}):")
    display(df_adv)
    lines = []
    lines.append(r"\begin{table}[htbp]")
    lines.append(r"\caption{Class-wise additional performance indicators of the proposed AGB-Net model (" + DATASET + r")}\label{tab:classwise_" + DATASET + "}")
    lines.append(r"\centering")
    lines.append(r"\begin{tabular}{@{}lcccc@{}}")
    lines.append(r"\toprule")
    lines.append(r"Class & FPR (\%) & FNR (\%) & FDR (\%) & FOR (\%) \\")
    lines.append(r"\midrule")
    for r in rows:
        lines.append(
            f"{r['Class']} & {r['FPR (%)']:.4f} & {r['FNR (%)']:.4f} & {r['FDR (%)']:.4f} & "
            f"{r['FOR (%)']:.4f} " + r"\\"
        )
    lines.append(r"\botrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")
    print("\n--- LaTeX ---")
    print("\n".join(lines))

## 13. Bar Chart: TNR, MCC, and NPV per Attack Class

In [27]:
if RUN_SINGLE_DEMO:
    class_labels = [l.split("(")[1].rstrip(")") for l in df_adv.index]
    tnr_vals = df_adv["TNR (%)"].values
    mcc_vals = df_adv["MCC"].values * 100
    npv_vals = df_adv["NPV (%)"].values
    x = np.arange(len(class_labels))
    w = 0.25
    fig, ax = plt.subplots(figsize=(max(14, len(class_labels) * 1.5), 6))
    b1 = ax.bar(x - w, tnr_vals, w, label="TNR (%)", color="mediumseagreen", edgecolor="white")
    b2 = ax.bar(x, mcc_vals, w, label="MCC (x100)", color="mediumpurple", edgecolor="white")
    b3 = ax.bar(x + w, npv_vals, w, label="NPV (%)", color="steelblue", edgecolor="white")
    for bars, vals in [(b1, tnr_vals), (b2, mcc_vals), (b3, npv_vals)]:
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
                    f"{val:.1f}", ha="center", va="bottom", fontsize=7.5)
    ax.set_xticks(x)
    ax.set_xticklabels(class_labels, rotation=30, ha="right", fontsize=10)
    ax.set_ylabel("Score", fontsize=11)
    ax.set_ylim(0, 115)
    ax.set_title(f"TNR, MCC (x100), and NPV per Attack Class - {DATASET}", fontsize=13, fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"tnr_mcc_npv_barchart_{DATASET}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 14. ROC Curve (One-vs-Rest, per Attack Class)

In [28]:
if RUN_SINGLE_DEMO:
    y_test_ovr = label_binarize(y_eval, classes=np.arange(NUM_CLASSES))
    y_prob_roc = best_model.predict(X_test_lstm, batch_size=BATCH, verbose=0)
    fpr_all, tpr_all = {}, {}
    for i in range(NUM_CLASSES):
        fpr_all[i], tpr_all[i], _ = roc_curve(y_test_ovr[:, i], y_prob_roc[:, i])
    all_fpr = np.unique(np.concatenate([fpr_all[i] for i in range(NUM_CLASSES)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(NUM_CLASSES):
        mean_tpr += np.interp(all_fpr, fpr_all[i], tpr_all[i])
    mean_tpr /= NUM_CLASSES
    macro_auc = auc(all_fpr, mean_tpr)
    palette = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))
    fig, ax = plt.subplots(figsize=(9, 7))
    for i, color in enumerate(palette):
        auc_i = auc(fpr_all[i], tpr_all[i])
        ax.plot(fpr_all[i], tpr_all[i], color=color, lw=1.2, alpha=0.55,
                label=f"{CLASS_ORDER[i]} (AUC={auc_i:.3f})")
    ax.plot(all_fpr, mean_tpr, color="black", lw=2.5, label=f"Macro-average (AUC={macro_auc:.4f})")
    ax.fill_between(all_fpr, mean_tpr, alpha=0.08, color="black")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title(f"AUC-ROC Curve -- {MODEL_NAME} on {DATASET}\nMacro-average AUC = {macro_auc:.4f}",
                 fontsize=13, fontweight="bold")
    ax.legend(loc="lower right", fontsize=8.5)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"auc_roc_curve_{DATASET}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Macro-average AUC: {macro_auc:.4f}")
    RESULTS.setdefault("performance", {})["macro_auc"] = float(macro_auc)
    save_results()

## 15. Computational Time Analysis

**Description:** warm-up

In [29]:
if RUN_SINGLE_DEMO:
    def measure_time(X_data):
        best_model.predict(X_data[:64], batch_size=BATCH, verbose=0)  
        t0 = time.perf_counter()
        best_model.predict(X_data, batch_size=BATCH, verbose=0)
        return time.perf_counter() - t0
    tr_sec  = measure_time(X_train_lstm)
    val_sec = measure_time(X_val_lstm)
    te_sec  = measure_time(X_test_lstm)
    total_sec = tr_sec + val_sec + te_sec
    cost_rows = [
        {"Phase": "Training",   "Time (s)": round(tr_sec, 2),  "Relative Cost (%)": round(tr_sec / total_sec * 100, 1)},
        {"Phase": "Validation", "Time (s)": round(val_sec, 2), "Relative Cost (%)": round(val_sec / total_sec * 100, 1)},
        {"Phase": "Testing",    "Time (s)": round(te_sec, 2),  "Relative Cost (%)": round(te_sec / total_sec * 100, 1)},
    ]
    df_cost = pd.DataFrame(cost_rows).set_index("Phase")
    print("Computational Time Analysis (Actual data, inference only):")
    display(df_cost)
    if RUN_SBOA:
        print(f"SBOA search time: {sboa_time:.1f}s | Final-model training time: {train_time_sec:.1f}s")
    RESULTS["timing"] = {
        "inference": {r["Phase"]: r["Time (s)"] for r in cost_rows},
        "train_time_sec": train_time_sec,
        "sboa_search_time_sec": sboa_time if RUN_SBOA else RESULTS.get("sboa", {}).get("search_time_sec"),
    }
    save_results()
    lines = []
    lines.append(r"\begin{table}[htbp]")
    lines.append(r"\caption{Computational time analysis for training, validation, and testing phases (" + DATASET + r")}\label{tab:t12_" + DATASET + "}")
    lines.append(r"\centering")
    lines.append(r"\begin{tabular}{@{}lcc@{}}")
    lines.append(r"\toprule")
    lines.append(r"Phase & Time (s) & Relative Cost (\%) \\")
    lines.append(r"\midrule")
    for r in cost_rows:
        lines.append(f"{r['Phase']} & {r['Time (s)']:.2f} & {r['Relative Cost (%)']:.1f} " + r"\\")
    lines.append(r"\botrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")
    print("\n--- LaTeX ---")
    print("\n".join(lines))

## 15b. Extra Experiments: Baselines, Fixed-vs-Adaptive Ablation, Top-k Sweep

`_load_split_preprocess` mirrors `run_pipeline`'s load/split/preprocess steps so every
extra experiment below sees identical data to the main model. `run_baseline_comparison`
re-implements BiGRU-only and GAT-BiGRU baselines (RQ1); `run_adaptive_ablation` trains
the same architecture with the adjacency fixed vs. learned (RQ2); `run_topk_sweep`
sweeps the graph neighbourhood size (RQ3). All three are invoked from the batch-runner
cell below, gated by the `RUN_BASELINES` / `RUN_ADAPTIVE_ABLATION` / `RUN_TOPK_SWEEP`
flags in the config cell.

**Description:** Incremental checkpoint after EVERY T value (not just at the end) so a crash on a

In [30]:
def _load_split_preprocess(dataset):
    """Shared loader used by the extra-experiment runners (ablation / top-k sweep / baselines)
    below: load -> optional stratified cap -> 70/15/15 split -> impute -> log1p -> scale ->
    variance-filter, fit strictly on the training split. Mirrors run_pipeline's steps 1-3
    exactly so every extra experiment sees identical splits/preprocessing to the main model."""
    X_raw, y_all, class_order, _ = LOADERS[dataset]()
    num_classes = len(class_order)
    cap = MAX_SAMPLES.get(dataset) if isinstance(MAX_SAMPLES, dict) else MAX_SAMPLES
    if cap and len(X_raw) > cap:
        X_raw, _, y_all, _ = train_test_split(X_raw, y_all, train_size=cap,
                                              random_state=SEED, stratify=y_all)
    X_tmp, X_test, y_tmp, y_test = train_test_split(X_raw, y_all, test_size=TEST_FRAC,
                                                    random_state=SEED, stratify=y_all)
    X_tr, X_val, y_tr, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_FRAC,
                                                random_state=SEED, stratify=y_tmp)
    imp = SimpleImputer(strategy="median")
    X_tr = imp.fit_transform(X_tr); X_val = imp.transform(X_val); X_test = imp.transform(X_test)
    def _addlog(X):
        return np.hstack([X, np.log1p(np.clip(X, 0, None))])
    X_tr, X_val, X_test = _addlog(X_tr), _addlog(X_val), _addlog(X_test)
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr); X_val = scaler.transform(X_val); X_test = scaler.transform(X_test)
    vt = VarianceThreshold(1e-4)
    X_tr = vt.fit_transform(X_tr); X_val = vt.transform(X_val); X_test = vt.transform(X_test)
    nf = X_tr.shape[1]
    _rs = lambda A: A.reshape(-1, TIMESTEPS, nf)
    X_tr_l, X_val_l, X_test_l = _rs(X_tr), _rs(X_val), _rs(X_test)
    corr_train = compute_corr_matrix(X_tr)
    freq = np.bincount(y_tr, minlength=num_classes).astype(float) / len(y_tr)
    cw = 1.0 / np.sqrt(freq + 1e-8); cw = np.clip(cw / cw.mean(), 0.5, 15.0)
    class_weights = dict(enumerate(cw))
    theta = (load_results(dataset).get("sboa", {}).get("theta_star")
             or DEFAULT_THETA_BY_DATASET.get(dataset, {"bigru1": 128, "bigru2": 64, "lr": LR,
                                                       "dropout": DROPOUT, "optimizer": "adam", "top_k": TOP_K}))
    return dict(
        X_tr_l=X_tr_l, X_val_l=X_val_l, X_test_l=X_test_l,
        y_tr=y_tr, y_val=y_val, y_test=y_test,
        nf=nf, num_classes=num_classes, class_order=class_order,
        corr_train=corr_train, class_weights=class_weights, theta=theta,
    )
def run_adaptive_ablation(dataset, epochs=None):
    """RQ2 -- Fixed-vs-adaptive-graph ablation: trains the identical architecture and
    SBOA-selected hyperparameters twice, once with a learnable adjacency (adaptive=True) and
    once with the adjacency frozen at its correlation-based initialization (adaptive=False),
    on identical splits. Directly tests the paper's central adaptive-graph novelty claim
    instead of only asserting it in prose."""
    import time as _time
    d = _load_split_preprocess(dataset)
    results_ab = {}
    for adaptive_flag in (True, False):
        tf.keras.backend.clear_session(); np.random.seed(SEED); tf.random.set_seed(SEED)
        theta = d["theta"]
        A = adjacency_from_corr(d["corr_train"], theta["top_k"])
        model = build_gcn_bigru(d["nf"], d["num_classes"], A, TIMESTEPS, dropout=theta["dropout"], reg=REG,
                                bigru1=theta["bigru1"], bigru2=theta["bigru2"],
                                top_k=theta["top_k"], adaptive=adaptive_flag)
        opt = OPTIMIZERS[theta["optimizer"]](learning_rate=theta["lr"], clipnorm=1.0)
        model.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        cbs = [EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, mode="min", verbose=0),
               ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-7, verbose=0)]
        _t0 = _time.perf_counter()
        model.fit(d["X_tr_l"], d["y_tr"], validation_data=(d["X_val_l"], d["y_val"]),
                  epochs=(epochs or EPOCHS), batch_size=BATCH, class_weight=d["class_weights"],
                  callbacks=cbs, verbose=2)
        elapsed = _time.perf_counter() - _t0
        p_test = model.predict(d["X_test_l"], batch_size=BATCH, verbose=0)
        pred = np.argmax(p_test, axis=1)
        tag = "adaptive" if adaptive_flag else "fixed"
        results_ab[tag] = {
            "accuracy": float(accuracy_score(d["y_test"], pred) * 100),
            "precision": float(precision_score(d["y_test"], pred, average="weighted", zero_division=0) * 100),
            "recall": float(recall_score(d["y_test"], pred, average="weighted", zero_division=0) * 100),
            "f1": float(f1_score(d["y_test"], pred, average="weighted", zero_division=0) * 100),
            "loss": float(log_loss(d["y_test"], p_test, labels=np.arange(d["num_classes"]))),
            "train_time_sec": elapsed,
        }
        print(f"[{dataset}] adaptive={adaptive_flag} -> "
              f"acc={results_ab[tag]['accuracy']:.2f}% f1={results_ab[tag]['f1']:.2f}%")
        if adaptive_flag:
            try:
                export_adjacency_comparison(model, d["corr_train"], theta["top_k"], dataset)
            except Exception as _e:
                print(f"[{dataset}] adjacency export skipped: {_e}")
    delta_acc = results_ab["adaptive"]["accuracy"] - results_ab["fixed"]["accuracy"]
    delta_f1 = results_ab["adaptive"]["f1"] - results_ab["fixed"]["f1"]
    results_ab["delta"] = {"accuracy": delta_acc, "f1": delta_f1}
    print(f"[{dataset}] Adaptive vs fixed graph: delta accuracy={delta_acc:+.2f}pp, delta f1={delta_f1:+.2f}pp")
    with open(RESULTS_DIR / f"ablation_adaptive_{dataset}.json", "w") as f:
        _json.dump(results_ab, f, indent=2, default=float)
    return results_ab
def run_topk_sweep(dataset, topk_values=(2, 4, 6, 8, 10, 12), epochs=25):
    """RQ3 -- Hyperparameter-sensitivity sweep over the graph neighbourhood size top-k, holding
    all other SBOA-selected hyperparameters fixed. Uses a reduced epoch budget (default 25,
    vs. the full EPOCHS) since this is a sensitivity study rather than the headline result."""
    d = _load_split_preprocess(dataset)
    theta = d["theta"]
    sweep_rows = []
    for k in topk_values:
        tf.keras.backend.clear_session(); np.random.seed(SEED); tf.random.set_seed(SEED)
        A = adjacency_from_corr(d["corr_train"], k)
        model = build_gcn_bigru(d["nf"], d["num_classes"], A, TIMESTEPS, dropout=theta["dropout"], reg=REG,
                                bigru1=theta["bigru1"], bigru2=theta["bigru2"],
                                top_k=k, adaptive=ADAPTIVE_GRAPH)
        opt = OPTIMIZERS[theta["optimizer"]](learning_rate=theta["lr"], clipnorm=1.0)
        model.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        es = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, mode="min", verbose=0)
        model.fit(d["X_tr_l"], d["y_tr"], validation_data=(d["X_val_l"], d["y_val"]), epochs=epochs,
                  batch_size=BATCH, class_weight=d["class_weights"], callbacks=[es], verbose=0)
        p_val = model.predict(d["X_val_l"], batch_size=BATCH, verbose=0)
        p_test = model.predict(d["X_test_l"], batch_size=BATCH, verbose=0)
        val_acc = float(accuracy_score(d["y_val"], np.argmax(p_val, 1)) * 100)
        test_acc = float(accuracy_score(d["y_test"], np.argmax(p_test, 1)) * 100)
        test_f1 = float(f1_score(d["y_test"], np.argmax(p_test, 1), average="weighted", zero_division=0) * 100)
        sweep_rows.append({"top_k": int(k), "val_accuracy": val_acc, "test_accuracy": test_acc, "test_f1": test_f1})
        print(f"[{dataset}] top_k={k:2d} -> val_acc={val_acc:.2f}% test_acc={test_acc:.2f}% test_f1={test_f1:.2f}%")
    df_sweep = pd.DataFrame(sweep_rows)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(df_sweep["top_k"], df_sweep["test_accuracy"], marker="o", label="Test Accuracy (%)")
    ax.plot(df_sweep["top_k"], df_sweep["test_f1"], marker="s", label="Test F1 (%)")
    ax.set_xlabel("Graph neighbourhood size (top-k)"); ax.set_ylabel("Score (%)")
    ax.set_title(f"Top-k sensitivity -- {dataset}", fontweight="bold")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"topk_sweep_{dataset}.png", dpi=150, bbox_inches="tight")
    plt.show()
    with open(RESULTS_DIR / f"topk_sweep_{dataset}.json", "w") as f:
        _json.dump(sweep_rows, f, indent=2, default=float)
    return df_sweep
def run_timesteps_sweep(dataset, timesteps_values=(10, 14, 18, 22, 26, 30), epochs=25):
    """RQ -- BiGRU sequence-length (TIMESTEPS) sensitivity sweep, holding all SBOA-selected
    hyperparameters fixed at the dataset's already-tuned theta* (mirrors run_topk_sweep's
    pattern exactly: only one architectural knob varies, no re-search).
    IMPORTANT CAVEAT: none of the 5 datasets preserve real temporal ordering through
    preprocessing -- every loader treats rows as i.i.d. flow records, and
    _load_split_preprocess does a stratified *shuffle* split. A "sequence" of length T here
    is formed by grouping T *consecutive rows after that shuffle split*; it is NOT a genuine
    temporal window of ordered network traffic. This sweep is a controlled
    architecture/capacity sensitivity study (does giving the BiGRU a longer, but
    semantically arbitrary, input window change accuracy/F1/speed?), not evidence of
    temporal patterns in the traffic.
    FAST-ITERATION SUBSETTING: this is an exploratory sweep, not the paper's headline
    numbers. Regardless of dataset size (and regardless of the MAX_SAMPLES cap used
    elsewhere), the flat training split is stratified-subsampled down to at most 5,000 rows
    -- and val/test down to at most 1,500 rows each -- BEFORE windowing, so every TIMESTEPS
    run trains on at most 5,000 // T sequences. If a split already has fewer rows than the
    cap, it is used as-is. Results from this sweep should be read as relative TIMESTEPS
    trends only, not publishable accuracy figures.
    Reuses _load_split_preprocess(dataset), which reshapes with the *global* TIMESTEPS
    constant (currently 1) -- so d["X_tr_l"]/d["X_val_l"]/d["X_test_l"] are effectively flat
    (N, 1, nf); flatten back to 2-D and re-window per T value tested here. d["corr_train"]
    (feature-feature correlation, computed on the pre-subsample training split) is reused
    unchanged for every T, matching run_topk_sweep -- it is unaffected by windowing and only
    weakly sensitive to the subsample, and recomputing it per T would add cost for no
    benefit here.
    """
    import time as _time
    d = _load_split_preprocess(dataset)
    theta = d["theta"]
    nf = d["nf"]
    num_classes = d["num_classes"]
    X_tr_flat = d["X_tr_l"].reshape(-1, nf)
    X_val_flat = d["X_val_l"].reshape(-1, nf)
    X_test_flat = d["X_test_l"].reshape(-1, nf)
    y_tr_flat = d["y_tr"]
    y_val_flat = d["y_val"]
    y_test_flat = d["y_test"]
    def _subsample(X, y, cap, seed):
        """Stratified subsample to at most `cap` rows; falls back to a plain random
        subsample if a class is too rare for sklearn's stratified split to place it in
        both the kept and discarded portions (e.g. a handful of Infiltration rows)."""
        if len(X) <= cap:
            return X, y
        try:
            X_sub, _, y_sub, _ = train_test_split(
                X, y, train_size=cap, random_state=seed, stratify=y)
        except ValueError:
            rng = np.random.default_rng(seed)
            idx = rng.choice(len(X), size=cap, replace=False)
            X_sub, y_sub = X[idx], y[idx]
        return X_sub, y_sub
    TRAIN_CAP, EVAL_CAP = 5000, 1500
    X_tr_flat, y_tr_flat = _subsample(X_tr_flat, y_tr_flat, TRAIN_CAP, SEED)
    X_val_flat, y_val_flat = _subsample(X_val_flat, y_val_flat, EVAL_CAP, SEED)
    X_test_flat, y_test_flat = _subsample(X_test_flat, y_test_flat, EVAL_CAP, SEED)
    print(f"[{dataset}] TIMESTEPS sweep on a fast-iteration subset: "
          f"train={len(X_tr_flat):,} val={len(X_val_flat):,} test={len(X_test_flat):,} rows")
    def _window(X_flat, y_flat, T):
        n_seq = len(X_flat) // T
        X_w = X_flat[: n_seq * T].reshape(n_seq, T, X_flat.shape[1])
        y_w = y_flat[T - 1: n_seq * T: T]
        return X_w, y_w
    A = adjacency_from_corr(d["corr_train"], theta["top_k"])
    sweep_rows = []
    for T in timesteps_values:
        X_tr_w, y_tr_w = _window(X_tr_flat, y_tr_flat, T)
        X_val_w, y_val_w = _window(X_val_flat, y_val_flat, T)
        X_test_w, y_test_w = _window(X_test_flat, y_test_flat, T)
        freq = np.bincount(y_tr_w, minlength=num_classes).astype(float) / len(y_tr_w)
        cw = 1.0 / np.sqrt(freq + 1e-8)
        cw = np.clip(cw / cw.mean(), 0.5, 15.0)
        class_weights = dict(enumerate(cw))
        tf.keras.backend.clear_session(); np.random.seed(SEED); tf.random.set_seed(SEED)
        model = build_gcn_bigru(nf, num_classes, A, T, dropout=theta["dropout"], reg=REG,
                                 bigru1=theta["bigru1"], bigru2=theta["bigru2"],
                                 top_k=theta["top_k"], adaptive=ADAPTIVE_GRAPH)
        opt = OPTIMIZERS[theta["optimizer"]](learning_rate=theta["lr"], clipnorm=1.0)
        model.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        es = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True,
                            mode="min", verbose=0)
        _t0 = _time.perf_counter()
        model.fit(X_tr_w, y_tr_w, validation_data=(X_val_w, y_val_w), epochs=epochs,
                  batch_size=BATCH, class_weight=class_weights, callbacks=[es], verbose=0)
        train_time_sec = _time.perf_counter() - _t0
        p_val = model.predict(X_val_w, batch_size=BATCH, verbose=0)
        p_test = model.predict(X_test_w, batch_size=BATCH, verbose=0)
        val_acc = float(accuracy_score(y_val_w, np.argmax(p_val, 1)) * 100)
        test_acc = float(accuracy_score(y_test_w, np.argmax(p_test, 1)) * 100)
        test_f1 = float(f1_score(y_test_w, np.argmax(p_test, 1), average="weighted",
                                  zero_division=0) * 100)
        row = {"timesteps": int(T), "val_accuracy": val_acc, "test_accuracy": test_acc,
               "test_f1": test_f1, "train_time_sec": train_time_sec,
               "n_train_seq": int(len(X_tr_w)), "n_val_seq": int(len(X_val_w)),
               "n_test_seq": int(len(X_test_w))}
        sweep_rows.append(row)
        print(f"[{dataset}] TIMESTEPS={T:2d} -> val_acc={val_acc:.2f}% test_acc={test_acc:.2f}% "
              f"test_f1={test_f1:.2f}% train_time={train_time_sec:.1f}s "
              f"(n_train_seq={row['n_train_seq']:,})")
        with open(RESULTS_DIR / f"timesteps_sweep_{dataset}.json", "w") as f:
            _json.dump(sweep_rows, f, indent=2, default=float)
    return pd.DataFrame(sweep_rows)
def run_baseline_comparison(dataset, epochs=None):
    """RQ1 -- Re-implements BiGRU-only (no graph) and GAT-BiGRU (attention-weighted feature
    graph) baselines on the SAME preprocessing, splits, class weights, and hardware as AGB-Net,
    giving a controlled comparison rather than citing accuracy numbers from other papers."""
    d = _load_split_preprocess(dataset)
    theta = d["theta"]
    mask = neighbor_mask_from_corr(d["corr_train"], theta["top_k"])
    def _fit_eval(model, tag):
        opt = OPTIMIZERS[theta["optimizer"]](learning_rate=theta["lr"], clipnorm=1.0)
        model.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        cbs = [EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, mode="min", verbose=0),
               ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-7, verbose=0)]
        import time as _time
        _t0 = _time.perf_counter()
        model.fit(d["X_tr_l"], d["y_tr"], validation_data=(d["X_val_l"], d["y_val"]), epochs=(epochs or EPOCHS),
                  batch_size=BATCH, class_weight=d["class_weights"], callbacks=cbs, verbose=2)
        elapsed = _time.perf_counter() - _t0
        p_test = model.predict(d["X_test_l"], batch_size=BATCH, verbose=0)
        pred = np.argmax(p_test, axis=1)
        row = {
            "accuracy": float(accuracy_score(d["y_test"], pred) * 100),
            "precision": float(precision_score(d["y_test"], pred, average="weighted", zero_division=0) * 100),
            "recall": float(recall_score(d["y_test"], pred, average="weighted", zero_division=0) * 100),
            "f1": float(f1_score(d["y_test"], pred, average="weighted", zero_division=0) * 100),
            "loss": float(log_loss(d["y_test"], p_test, labels=np.arange(d["num_classes"]))),
            "train_time_sec": elapsed, "params": int(model.count_params()),
        }
        print(f"[{dataset}] {tag} -> acc={row['accuracy']:.2f}% f1={row['f1']:.2f}% "
              f"params={row['params']:,} time={elapsed:.1f}s")
        return row
    baselines = {}
    tf.keras.backend.clear_session(); np.random.seed(SEED); tf.random.set_seed(SEED)
    baselines["BiGRU_only"] = _fit_eval(
        build_bigru_only(d["nf"], d["num_classes"], TIMESTEPS, dropout=theta["dropout"], reg=REG,
                         bigru1=theta["bigru1"], bigru2=theta["bigru2"]), "BiGRU-only")
    tf.keras.backend.clear_session(); np.random.seed(SEED); tf.random.set_seed(SEED)
    baselines["GAT_BiGRU"] = _fit_eval(
        build_gat_bigru(d["nf"], d["num_classes"], mask, TIMESTEPS, dropout=theta["dropout"], reg=REG,
                        bigru1=theta["bigru1"], bigru2=theta["bigru2"]), "GAT-BiGRU")
    with open(RESULTS_DIR / f"baselines_{dataset}.json", "w") as f:
        _json.dump(baselines, f, indent=2, default=float)
    return baselines

## 16. Batch Runner -- Evaluate Every Dataset, Fully Unattended

`DATASETS_TO_RUN` (config cell) now defaults to **all five** datasets. Every dataset is processed
in a **fully independent pass** (its own load -> preprocess -> feature graph -> SBOA -> final
training -> evaluation), so runs never share state, and one dataset's failure does not stop the
rest of the batch (each stage below is wrapped in try/except).

For each dataset:
* if `results/results_<dataset>.json` already holds SBOA-selected hyperparameters, the SBOA
  search is **skipped** and those parameters are reused (this is the default: `RUN_SBOA=False`
  reuses the real cached/default theta* for every dataset, so only final-model training and
  evaluation are re-run, not the multi-hour search);
* otherwise, if `RUN_SBOA=True` a fresh SBOA search runs and is cached; if `RUN_SBOA=False` and
  nothing is cached, the per-dataset default hyperparameters are used.

Outputs per dataset, all persisted to `results/results_<dataset>.json`: the performance table,
exact train/val/test split sizes, confusion-matrix counts and percentages, per-class
FPR/FNR/FDR/FOR/TNR/MCC/NPV, one-vs-rest ROC and precision-recall curves (downsampled to a fixed
200-point grid), inference timing, and a learned-vs-initial adjacency comparison (heatmaps +
edge-change statistics, `adjacency_<dataset>.json`).

After the main batch, three further unattended stages run (each gated by its own config flag and
dataset list): `run_baseline_comparison` (RQ1: re-implemented BiGRU-only / GAT-BiGRU baselines,
`results/baselines_<dataset>.json`), `run_adaptive_ablation` (RQ2: fixed-vs-adaptive-graph
ablation, `results/ablation_adaptive_<dataset>.json`), and `run_topk_sweep` (RQ3: top-k
neighbourhood-size sensitivity, `results/topk_sweep_<dataset>.json`).


**Description:** 1. Load (+ optional stratified cap) --------------------------------------------------

In [31]:
def _downsample_curve(x_grid_source, y_source, n_points=200):
    """Interpolate a curve onto a fixed-size grid so ROC/PR arrays stay compact in the JSON
    results file regardless of test-set size (sklearn's roc_curve/precision_recall_curve
    return one point per unique score, which can be very large for big test sets)."""
    grid = np.linspace(0.0, 1.0, n_points)
    y_interp = np.interp(grid, x_grid_source, y_source)
    return grid.tolist(), y_interp.tolist()
def run_pipeline(dataset, run_sboa=None, max_samples="config", make_plots=True):
    """Fully self-contained, independent evaluation of one dataset.
    Reuses the shared helpers defined above (LOADERS, compute_corr_matrix, adjacency_from_corr,
    build_gcn_bigru, sboa_fitness, run_sboa) but keeps all per-dataset state local, so several
    datasets can be evaluated back-to-back without one run affecting another.
    In addition to the headline performance table, this now also computes and persists (for
    every dataset, not just the two primary ones): confusion-matrix counts, per-class
    FPR/FNR/FDR/FOR/TNR/MCC/NPV, one-vs-rest ROC and precision-recall curves (downsampled to a
    fixed grid), inference timing, exact train/val/test split sizes, and a comparison of the
    learned adjacency against its correlation-graph initialization."""
    import time as _time
    assert dataset in DATASETS_ALL, f"Unknown dataset: {dataset}"
    if run_sboa is None:
        run_sboa = RUN_SBOA
    if max_samples == "config":
        max_samples = MAX_SAMPLES.get(dataset) if isinstance(MAX_SAMPLES, dict) else MAX_SAMPLES
    print("\n" + "=" * 72 + f"\n RUN_PIPELINE :: {dataset}\n" + "=" * 72)
    t_pipe = _time.perf_counter()
    X_raw, y_all, class_order, _ = LOADERS[dataset]()
    num_classes = len(class_order)
    print(f"[{dataset}] raw features={X_raw.shape[1]} samples={X_raw.shape[0]:,} classes={num_classes}")
    if max_samples and len(X_raw) > max_samples:
        X_raw, _, y_all, _ = train_test_split(X_raw, y_all, train_size=max_samples,
                                              random_state=SEED, stratify=y_all)
        print(f"[{dataset}] stratified cap -> {max_samples:,} rows")
    X_tmp, X_test, y_tmp, y_test = train_test_split(X_raw, y_all, test_size=TEST_FRAC,
                                                    random_state=SEED, stratify=y_all)
    X_tr, X_val, y_tr, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_FRAC,
                                                random_state=SEED, stratify=y_tmp)
    split_sizes = {"train": int(len(X_tr)), "val": int(len(X_val)), "test": int(len(X_test)),
                   "total": int(len(X_raw))}
    imp = SimpleImputer(strategy="median")
    X_tr = imp.fit_transform(X_tr); X_val = imp.transform(X_val); X_test = imp.transform(X_test)
    def _addlog(X): return np.hstack([X, np.log1p(np.clip(X, 0, None))])
    X_tr, X_val, X_test = _addlog(X_tr), _addlog(X_val), _addlog(X_test)
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr); X_val = scaler.transform(X_val); X_test = scaler.transform(X_test)
    vt = VarianceThreshold(1e-4)
    X_tr = vt.fit_transform(X_tr); X_val = vt.transform(X_val); X_test = vt.transform(X_test)
    nf = X_tr.shape[1]
    _rs = lambda A: A.reshape(-1, TIMESTEPS, nf)
    X_tr_l, X_val_l, X_test_l = _rs(X_tr), _rs(X_val), _rs(X_test)
    print(f"[{dataset}] features(after log1p+varfilter)={nf} | "
          f"train={len(X_tr):,} val={len(X_val):,} test={len(X_test):,}")
    corr_train = compute_corr_matrix(X_tr)
    freq = np.bincount(y_tr, minlength=num_classes).astype(float) / len(y_tr)
    cw = 1.0 / np.sqrt(freq + 1e-8); cw = np.clip(cw / cw.mean(), 0.5, 15.0)
    class_weights = dict(enumerate(cw))
    res = load_results(dataset)
    cached_theta = res.get("sboa", {}).get("theta_star")
    convergence = res.get("sboa", {}).get("convergence", [])
    if cached_theta:
        theta_star = cached_theta
        print(f"[{dataset}] cached SBOA hyperparameters found -> skipping search. theta*={theta_star}")
    elif run_sboa:
        print(f"[{dataset}] no cached SBOA params -> running SBOA (P={SBOA_POP}, T={SBOA_ITERS})")
        n_sub = min(max(int(len(X_tr) * SBOA_SAMPLE_FRAC), BATCH * 4), len(X_tr))
        si = np.random.default_rng(SEED).choice(len(X_tr), size=n_sub, replace=False)
        Xs, ys = X_tr[si], y_tr[si]
        Xstr, Xsva, ystr, ysva = train_test_split(Xs, ys, test_size=0.2, random_state=SEED, stratify=ys)
        corr_sboa = compute_corr_matrix(Xstr)
        Xstr_l = Xstr.reshape(-1, TIMESTEPS, nf); Xsva_l = Xsva.reshape(-1, TIMESTEPS, nf)
        def _fit(v):
            return sboa_fitness(v, Xstr_l, ystr, Xsva_l, ysva, corr_sboa, num_classes, nf, TIMESTEPS)
        _t0 = _time.perf_counter()
        _, theta_star, best_f, convergence = run_sboa(_fit, dim=DIM, pop_size=SBOA_POP,
                                                      iters=SBOA_ITERS, seed=SEED)
        res["sboa"] = {"theta_star": theta_star, "best_val_accuracy": best_f, "convergence": convergence,
                       "search_time_sec": _time.perf_counter() - _t0, "pop_size": SBOA_POP,
                       "iters": SBOA_ITERS, "sample_frac": SBOA_SAMPLE_FRAC, "epochs_per_candidate": SBOA_EPOCHS}
        print(f"[{dataset}] SBOA done (val_acc={best_f*100:.2f}%). theta*={theta_star}")
    else:
        theta_star = DEFAULT_THETA_BY_DATASET.get(dataset, {"bigru1": 128, "bigru2": 64, "lr": LR,
                                                       "dropout": DROPOUT, "optimizer": "adam", "top_k": TOP_K})
        print(f"[{dataset}] RUN_SBOA=False and no cache -> using default theta*={theta_star}")
    A_final = adjacency_from_corr(corr_train, theta_star["top_k"])
    tf.keras.backend.clear_session(); np.random.seed(SEED); tf.random.set_seed(SEED)
    model = build_gcn_bigru(nf, num_classes, A_final, TIMESTEPS, dropout=theta_star["dropout"], reg=REG,
                            bigru1=theta_star["bigru1"], bigru2=theta_star["bigru2"],
                            top_k=theta_star["top_k"], adaptive=ADAPTIVE_GRAPH)
    opt = OPTIMIZERS[theta_star["optimizer"]](learning_rate=theta_star["lr"], clipnorm=1.0)
    model.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    cbs = [EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, mode="min", verbose=1),
           ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-7, verbose=1)]
    _t0 = _time.perf_counter()
    history = model.fit(X_tr_l, y_tr, validation_data=(X_val_l, y_val), epochs=EPOCHS, batch_size=BATCH,
                        class_weight=class_weights, callbacks=cbs, verbose=2)
    train_time = _time.perf_counter() - _t0
    def _ev(Xs, ys):
        p = model.predict(Xs, batch_size=BATCH, verbose=0); pr = np.argmax(p, 1)
        return (accuracy_score(ys, pr) * 100, precision_score(ys, pr, average="weighted", zero_division=0) * 100,
                recall_score(ys, pr, average="weighted", zero_division=0) * 100,
                f1_score(ys, pr, average="weighted", zero_division=0) * 100,
                log_loss(ys, p, labels=np.arange(num_classes)))
    perf = {"Training": _ev(X_tr_l, y_tr), "Validation": _ev(X_val_l, y_val), "Testing": _ev(X_test_l, y_test)}
    perf_df = pd.DataFrame([{"Dataset Split": k, "Accuracy (%)": round(v[0], 2), "Precision (%)": round(v[1], 2),
                             "Recall (%)": round(v[2], 2), "F1-score (%)": round(v[3], 2), "Loss": round(v[4], 4)}
                            for k, v in perf.items()]).set_index("Dataset Split")
    print(f"[{dataset}] Performance ({MODEL_NAME}):")
    try:
        display(perf_df)
    except NameError:
        print(perf_df)
    res["performance"] = {k: {"accuracy": v[0], "precision": v[1], "recall": v[2], "f1": v[3], "loss": v[4]}
                          for k, v in perf.items()}
    res["config"] = {"adaptive_graph": ADAPTIVE_GRAPH, "theta_star": theta_star, "num_classes": num_classes,
                     "num_features": nf, "class_order": class_order, "train_time_sec": train_time,
                     "max_samples": max_samples, "split_sizes": split_sizes}
    y_prob_test = model.predict(X_test_l, batch_size=BATCH, verbose=0)
    y_pred = np.argmax(y_prob_test, axis=1)
    cm = confusion_matrix(y_test, y_pred, labels=np.arange(num_classes)).astype(float)
    cm_pct = cm / (cm.sum(1, keepdims=True) + 1e-9) * 100
    classwise_rows = []
    for i, label in enumerate(class_order):
        y_tb = (y_test == i).astype(int)
        y_pb = (y_pred == i).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_tb, y_pb, labels=[0, 1]).ravel()
        fpr_c = fp / (fp + tn) * 100 if (fp + tn) > 0 else 0.0
        fnr_c = fn / (fn + tp) * 100 if (fn + tp) > 0 else 0.0
        fdr_c = fp / (fp + tp) * 100 if (fp + tp) > 0 else 0.0
        for_c = fn / (fn + tn) * 100 if (fn + tn) > 0 else 0.0
        tnr_c = tn / (tn + fp) * 100 if (tn + fp) > 0 else 0.0
        npv_c = tn / (tn + fn) * 100 if (tn + fn) > 0 else 0.0
        mcc_c = (matthews_corrcoef(y_tb, y_pb)
                 if len(np.unique(y_tb)) > 1 and len(np.unique(y_pb)) > 1 else 0.0)
        classwise_rows.append({
            "class": label, "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
            "fpr": round(float(fpr_c), 4), "fnr": round(float(fnr_c), 4),
            "fdr": round(float(fdr_c), 4), "for_": round(float(for_c), 4),
            "tnr": round(float(tnr_c), 4), "mcc": round(float(mcc_c), 4), "npv": round(float(npv_c), 4),
        })
    y_test_ovr = label_binarize(y_test, classes=np.arange(num_classes))
    if num_classes == 2:
        y_test_ovr = np.hstack([1 - y_test_ovr, y_test_ovr])
    roc_data, pr_data = {}, {}
    for i, label in enumerate(class_order):
        fpr_i, tpr_i, _ = roc_curve(y_test_ovr[:, i], y_prob_test[:, i])
        auc_i = float(auc(fpr_i, tpr_i))
        grid, tpr_ds = _downsample_curve(fpr_i, tpr_i)
        roc_data[label] = {"fpr": grid, "tpr": tpr_ds, "auc": auc_i}
        prec_i, rec_i, _ = precision_recall_curve(y_test_ovr[:, i], y_prob_test[:, i])
        order = np.argsort(rec_i)
        grid_r, prec_ds = _downsample_curve(rec_i[order], prec_i[order])
        pr_data[label] = {"recall": grid_r, "precision": prec_ds}
    all_fpr = np.unique(np.concatenate([np.array(roc_data[l]["fpr"]) for l in class_order]))
    mean_tpr = np.mean([np.interp(all_fpr, roc_data[l]["fpr"], roc_data[l]["tpr"]) for l in class_order], axis=0)
    macro_auc = float(auc(all_fpr, mean_tpr))
    roc_data["macro_average"] = {"fpr": all_fpr.tolist(), "tpr": mean_tpr.tolist(), "auc": macro_auc}
    print(f"[{dataset}] Macro-average AUC: {macro_auc:.4f}")
    if make_plots:
        fig, ax = plt.subplots(figsize=(max(8, num_classes * 1.1), max(6, num_classes * 0.9)))
        sns.heatmap(cm_pct, annot=np.array([[f"{v:.1f}" for v in r] for r in cm_pct]), fmt="", cmap="Blues",
                    vmin=0, vmax=100, xticklabels=class_order, yticklabels=class_order, ax=ax,
                    cbar_kws={"label": "% of True Class"}, linewidths=0.5, linecolor="white")
        ax.set_title(f"Confusion Matrix - {MODEL_NAME} ({dataset})", fontweight="bold")
        ax.set_xlabel("Predicted Label"); ax.set_ylabel("True Label"); ax.tick_params(axis="x", rotation=30)
        plt.tight_layout(); plt.savefig(f"confusion_matrix_{dataset}.png", dpi=150, bbox_inches="tight"); plt.show()
        palette = plt.cm.tab10(np.linspace(0, 1, num_classes))
        fig, ax = plt.subplots(figsize=(9, 7))
        for i, color in zip(range(num_classes), palette):
            ax.plot(roc_data[class_order[i]]["fpr"], roc_data[class_order[i]]["tpr"], color=color, lw=1.2,
                    alpha=0.6, label=f"{class_order[i]} (AUC={roc_data[class_order[i]]['auc']:.3f})")
        ax.plot(all_fpr, mean_tpr, color="black", lw=2.5, label=f"Macro-average (AUC={macro_auc:.4f})")
        ax.plot([0, 1], [0, 1], "k--", lw=1)
        ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
        ax.set_title(f"AUC-ROC Curve -- {MODEL_NAME} on {dataset}\nMacro-average AUC = {macro_auc:.4f}",
                     fontweight="bold")
        ax.legend(loc="lower right", fontsize=8); ax.grid(True, alpha=0.3)
        plt.tight_layout(); plt.savefig(f"auc_roc_curve_{dataset}.png", dpi=150, bbox_inches="tight"); plt.show()
        try:
            export_adjacency_comparison(model, corr_train, theta_star["top_k"], dataset)
        except Exception as _e:
            print(f"[{dataset}] adjacency export skipped: {_e}")
    def _measure_time(Xd):
        model.predict(Xd[:64], batch_size=BATCH, verbose=0)
        t0 = _time.perf_counter(); model.predict(Xd, batch_size=BATCH, verbose=0)
        return _time.perf_counter() - t0
    tr_sec, val_sec, te_sec = _measure_time(X_tr_l), _measure_time(X_val_l), _measure_time(X_test_l)
    res["confusion_matrix"] = {"labels": class_order, "counts": cm.astype(int).tolist(),
                               "percent": cm_pct.tolist()}
    res["classwise_indicators"] = classwise_rows
    res["roc_curves"] = roc_data
    res["pr_curves"] = pr_data
    res["timing"] = {"inference": {"Training": tr_sec, "Validation": val_sec, "Testing": te_sec},
                     "train_time_sec": train_time,
                     "sboa_search_time_sec": res.get("sboa", {}).get("search_time_sec")}
    with open(_results_path(dataset), "w") as f:
        _json.dump(res, f, indent=2, default=float)
    print(f"[{dataset}] saved -> {_results_path(dataset)} | pipeline time {_time.perf_counter() - t_pipe:.0f}s")
    return perf, theta_star

**Description:** Run every dataset in DATASETS_TO_RUN, one fully independent pass each, fully unattended.

In [ ]:
import time as _time
_batch_summary = {}
for _ds in DATASETS_TO_RUN:
    try:
        if GPUS:
            tf.keras.backend.clear_session()
            for _gpu in GPUS:
                try:
                    tf.config.experimental.reset_memory_stats(_gpu)
                except:
                    pass
            _time.sleep(0.5)  
        _perf, _theta = run_pipeline(_ds)
        _batch_summary[_ds] = _perf["Testing"]
    except Exception as _e:
        import traceback; traceback.print_exc()
        _batch_summary[_ds] = f"ERROR: {_e}"
print("\n" + "=" * 72 + "\n BATCH SUMMARY (test split)\n" + "=" * 72)
print(f"{'dataset':12s} {'acc':>8} {'prec':>8} {'rec':>8} {'f1':>8}")
for _ds, _v in _batch_summary.items():
    if isinstance(_v, tuple):
        print(f"{_ds:12s} {_v[0]:8.2f} {_v[1]:8.2f} {_v[2]:8.2f} {_v[3]:8.2f}")
    else:
        print(f"{_ds:12s} {_v}")
if RUN_BASELINES:
    print("\n" + "=" * 72 + "\n BASELINE COMPARISON (RQ1)\n" + "=" * 72)
    _baseline_summary = {}
    for _ds in BASELINE_DATASETS:
        try:
            _baseline_summary[_ds] = run_baseline_comparison(_ds)
        except Exception as _e:
            import traceback; traceback.print_exc()
            _baseline_summary[_ds] = f"ERROR: {_e}"
if RUN_ADAPTIVE_ABLATION:
    print("\n" + "=" * 72 + "\n FIXED-VS-ADAPTIVE-GRAPH ABLATION (RQ2)\n" + "=" * 72)
    _ablation_summary = {}
    for _ds in ABLATION_DATASETS:
        try:
            _ablation_summary[_ds] = run_adaptive_ablation(_ds)
        except Exception as _e:
            import traceback; traceback.print_exc()
            _ablation_summary[_ds] = f"ERROR: {_e}"
if RUN_TOPK_SWEEP:
    print("\n" + "=" * 72 + "\n TOP-K SENSITIVITY SWEEP (RQ3)\n" + "=" * 72)
    _sweep_summary = {}
    for _ds in SWEEP_DATASETS:
        try:
            _sweep_summary[_ds] = run_topk_sweep(_ds, topk_values=TOPK_SWEEP_VALUES, epochs=TOPK_SWEEP_EPOCHS)
        except Exception as _e:
            import traceback; traceback.print_exc()
            _sweep_summary[_ds] = f"ERROR: {_e}"
print("\n" + "=" * 72 + "\n ALL REQUESTED EXPERIMENTS COMPLETE\n" + "=" * 72)


 RUN_PIPELINE :: cicids
[cicids] raw features=78 samples=2,522,362 classes=8
[cicids] features(after log1p+varfilter)=140 | train=1,766,661 val=377,346 test=378,355
[cicids] RUN_SBOA=False and no cache -> using default theta*={'bigru1': 32, 'bigru2': 84, 'lr': 0.00024017660564659387, 'dropout': 0.10044824369377355, 'optimizer': 'adam', 'top_k': 6}
Epoch 1/100
3451/3451 - 119s - loss: 0.1921 - accuracy: 0.9893 - val_loss: 0.0401 - val_accuracy: 0.9969 - lr: 2.4018e-04 - 119s/epoch - 35ms/step
Epoch 2/100
3451/3451 - 108s - loss: 0.0234 - accuracy: 0.9967 - val_loss: 0.0257 - val_accuracy: 0.9952 - lr: 2.4018e-04 - 108s/epoch - 31ms/step
Epoch 3/100
3451/3451 - 116s - loss: 0.0136 - accuracy: 0.9971 - val_loss: 0.0146 - val_accuracy: 0.9971 - lr: 2.4018e-04 - 116s/epoch - 34ms/step
Epoch 4/100
3451/3451 - 114s - loss: 0.0109 - accuracy: 0.9973 - val_loss: 0.0141 - val_accuracy: 0.9973 - lr: 2.4018e-04 - 114s/epoch - 33ms/step
Epoch 5/100
3451/3451 - 116s - loss: 0.0097 - accuracy: 0.997

,Accuracy (%),Precision (%),Recall (%),F1-score (%),Loss
Dataset Split,,,,,
Training,99.85,99.85,99.85,99.84,0.0054
Validation,99.85,99.85,99.85,99.84,0.0056
Testing,99.84,99.84,99.84,99.83,0.0057


[cicids] Macro-average AUC: 0.9989


C:\Users\ashis\AppData\Local\Temp\ipykernel_7544\135843759.py:189: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(f"confusion_matrix_{dataset}.png", dpi=150, bbox_inches="tight"); plt.show()
C:\Users\ashis\AppData\Local\Temp\ipykernel_7544\135843759.py:202: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(f"auc_roc_curve_{dataset}.png", dpi=150, bbox_inches="tight"); plt.show()


[cicids] Adjacency change vs. initial correlation graph: kept=404, added=0, removed=546, Frobenius dist=5.183, edge-weight corr=0.375


C:\Users\ashis\AppData\Local\Temp\ipykernel_7544\2186285210.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


[cicids] saved -> results\results_cicids.json | pipeline time 11114s

 BATCH SUMMARY (test split)
dataset           acc     prec      rec       f1
cicids          99.84    99.84    99.84    99.83

 BASELINE COMPARISON (RQ1)
Epoch 1/100
3451/3451 - 120s - loss: 0.1667 - accuracy: 0.9839 - val_loss: 0.0299 - val_accuracy: 0.9971 - lr: 2.4018e-04 - 120s/epoch - 35ms/step
Epoch 2/100
3451/3451 - 102s - loss: 0.0187 - accuracy: 0.9966 - val_loss: 0.0177 - val_accuracy: 0.9972 - lr: 2.4018e-04 - 102s/epoch - 30ms/step
Epoch 3/100
3451/3451 - 99s - loss: 0.0129 - accuracy: 0.9968 - val_loss: 0.0145 - val_accuracy: 0.9974 - lr: 2.4018e-04 - 99s/epoch - 29ms/step
Epoch 4/100
3451/3451 - 92s - loss: 0.0113 - accuracy: 0.9969 - val_loss: 0.0127 - val_accuracy: 0.9977 - lr: 2.4018e-04 - 92s/epoch - 27ms/step
Epoch 5/100
3451/3451 - 92s - loss: 0.0107 - accuracy: 0.9970 - val_loss: 0.0125 - val_accuracy: 0.9975 - lr: 2.4018e-04 - 92s/epoch - 27ms/step
Epoch 6/100
3451/3451 - 92s - loss: 0.0103 - ac

## 17. Temporal-Windowing Validation (does a *real* BiGRU sequence help?)

The `run_timesteps_sweep` above deliberately builds "sequences" by grouping
consecutive rows **after a stratified shuffle split**, so its windows carry no
temporal signal -- which is exactly why longer windows there only add label
noise and shrink the sample count, making accuracy fall as T grows. This
section replaces that with a *genuine* temporal protocol and runs the
experiments needed to decide whether temporal modelling actually helps on
these datasets:

**Protocol changes**
1. **Keep time + host metadata.** Loaders here retain `Timestamp`/`ts`/
   `frame.time` and a host key (`Source IP`/`src_ip`/`ip.src_host`) as
   *metadata only* -- never as model features.
2. **Chronological split** (earliest 70% train / next 15% val / last 15%
   test) instead of a stratified shuffle -- the standard leak-free protocol
   for windowed IDS data.
3. **Per-host sliding windows, stride 1.** Windows are built inside each host,
   ordered by time, and never cross a host or split boundary. Stride 1 keeps
   ~N windows from N rows, so the sample-count collapse that plagued the old
   sweep disappears.
4. **Label = last row in the window** ("classify this flow given the host's
   recent history"), keeping evaluation comparable to the T=1 per-flow setup.

**Experiments run here**
- **E1 -- Temporal vs. destroyed-temporal control (the decisive test).** For
  each T, train twice on *identical windows*: once in true time order, once
  with each host's rows shuffled before windowing. Same window count, same
  capacity, same label mix -- the *only* difference is whether order is real.
  If real order beats the shuffled control (and beats T=1), the gain is
  genuinely temporal and not just capacity/label-smoothing.
- **E2 -- Corrected T sweep.** Test accuracy/F1 vs T under the real protocol,
  to see whether "more timesteps help" now holds.
- **E3 -- Apples-to-apples headline check (optional).** Non-overlapping
  (stride=T) windows with a *window-level* stratified shuffle split -- no row
  leakage -- so T>1 can be compared against the original shuffled-split
  headline numbers on the same split style.

Datasets without any timestamp (NSL-KDD; the pooled CIC-IoT-2023 CSVs) are
skipped automatically and should stay at T=1 in the paper, stated explicitly.


In [ ]:
# === Temporal-windowing validation: config + helpers ==========================
RUN_TEMPORAL_VALIDATION = True
TEMPORAL_DATASETS       = ["cicids", "ton_iot", "edge_iiot"]   # have timestamps
TEMPORAL_T_VALUES       = [1, 4, 8, 16, 32]
TEMPORAL_EPOCHS         = 20
TEMPORAL_ROW_CAP        = 60_000     # group-aware cap; keeps whole hosts intact
RUN_TEMPORAL_HEADLINE   = True       # E3 apples-to-apples check
TEMPORAL_HEADLINE_T     = 16

import time as _t_time

# --- metadata-aware loaders (features identical to the originals; the only
#     addition is a meta DataFrame of ['ts','group'] used purely for
#     ordering/grouping, never fed to the model) ------------------------------
def _tl_cicids():
    D = find_dataset_root() / "cicids"
    pq = D / "cicids2017_combined.parquet"
    if pq.exists():
        df = pd.read_parquet(pq).drop_duplicates()
    else:
        raw = sorted(D.glob("*.pcap_ISCX.csv"))
        if not raw:
            raise FileNotFoundError(f"No CICIDS2017 CSVs under {D}")
        fr = []
        for f in raw:
            d = pd.read_csv(f, low_memory=False); d.columns = [c.strip() for c in d.columns]
            fr.append(d)
        df = pd.concat(fr, ignore_index=True).drop_duplicates()
    def map_attack(v):
        vl = str(v).strip().lower()
        if vl == "benign": return "Normal"
        if vl.startswith("web attack"): return "Web Attack"
        if vl in ("ftp-patator", "ssh-patator"): return "Brute Force"
        if vl.startswith("dos") or vl == "heartbleed": return "DoS"
        if vl == "ddos": return "DDoS"
        if vl == "portscan": return "PortScan"
        if vl == "bot": return "Bot(s)"
        if vl == "infiltration": return "Infiltration"
        return "Other"
    attack = df["Label"].astype(str).map(map_attack)
    le = LabelEncoder(); y = le.fit_transform(attack); class_order = list(le.classes_)
    bot = next((l for l in class_order if "bot" in l.lower()), None)
    if bot and class_order.index(bot) != len(class_order) - 1:
        old = list(le.classes_); class_order.remove(bot); class_order.append(bot)
        remap = {i: class_order.index(n) for i, n in enumerate(old)}
        y = np.array([remap[v] for v in y])
    ts_col = next((c for c in ["Timestamp", "timestamp"] if c in df.columns), None)
    grp_col = next((c for c in ["Source IP", "Src IP", "src_ip"] if c in df.columns), None)
    ts = pd.to_datetime(df[ts_col], errors="coerce", dayfirst=True)
    group = df[grp_col].astype(str) if grp_col else pd.Series(["_all_"] * len(df))
    drop_cols = [c for c in ["Label", "Flow ID", "Source IP", "Destination IP",
                             "Timestamp", "Src IP", "Dst IP"] if c in df.columns]
    X = (df.drop(columns=drop_cols).select_dtypes(include=[np.number])
           .replace([np.inf, -np.inf], np.nan).values.astype(np.float32))
    meta = pd.DataFrame({"ts": ts.values, "group": group.values})
    return X, y.astype(np.int32), class_order, meta

def _tl_tabular(dataset):
    if dataset == "ton_iot":
        D = find_dataset_root() / "ton_iot"
        df = pd.read_csv(D / "train_test_network_full.csv")
        label_col, ts_col, grp_col = "type", "ts", "src_ip"
        drop_id = ["src_ip", "dst_ip", "dns_query", "ssl_subject", "ssl_issuer",
                   "http_uri", "http_user_agent", "label", "ts"]   # ts now meta-only
        ts = pd.to_datetime(pd.to_numeric(df[ts_col], errors="coerce"), unit="s", errors="coerce")
    elif dataset == "edge_iiot":
        D = find_dataset_root() / "Edge_iiot"
        df = pd.read_csv(D / "DNN-EdgeIIoT-dataset.csv", low_memory=False)
        label_col, ts_col, grp_col = "Attack_type", "frame.time", "ip.src_host"
        drop_id = ["frame.time", "ip.src_host", "ip.dst_host", "arp.src.proto_ipv4",
                   "arp.dst.proto_ipv4", "http.request.full_uri", "http.request.uri.query",
                   "http.file_data", "http.referer", "tcp.options", "tcp.payload",
                   "mqtt.msg", "dns.qry.name", "Attack_label"]
        ts = pd.to_datetime(df[ts_col], errors="coerce")
    else:
        raise ValueError(dataset)
    group = df[grp_col].astype(str) if grp_col in df.columns else pd.Series(["_all_"] * len(df))
    X, y, class_order = _prep_tabular(df, label_col, drop_id, onehot_cap=50)
    meta = pd.DataFrame({"ts": ts.values, "group": group.values})
    return X, y.astype(np.int32), class_order, meta

def temporal_load(dataset):
    """Returns (X, y, class_order, meta) or None if the dataset has no usable
    timestamp (NSL-KDD, pooled CIC-IoT-2023)."""
    if dataset == "cicids":
        return _tl_cicids()
    if dataset in ("ton_iot", "edge_iiot"):
        return _tl_tabular(dataset)
    return None   # no temporal metadata available

# --- group-aware row cap (keeps whole hosts so sequences stay intact) --------
def _cap_by_groups(X, y, meta, cap, seed):
    if cap is None or len(X) <= cap:
        return X, y, meta.reset_index(drop=True)
    counts = meta["group"].value_counts()
    uniq = counts.index.to_numpy().copy()
    np.random.default_rng(seed).shuffle(uniq)
    keep, tot = [], 0
    for g in uniq:
        keep.append(g); tot += int(counts[g])
        if tot >= cap:
            break
    m = meta["group"].isin(set(keep)).values
    return X[m], y[m], meta[m].reset_index(drop=True)

# --- preprocessing fit strictly on the training rows (mirrors section 2) -----
def _make_prep(X_fit):
    imp = SimpleImputer(strategy="median").fit(X_fit)
    def addlog(X): return np.hstack([X, np.log1p(np.clip(X, 0, None))])
    Xi = addlog(imp.transform(X_fit))
    sc = StandardScaler().fit(Xi)
    vt = VarianceThreshold(1e-4).fit(sc.transform(Xi))
    def f(X): return vt.transform(sc.transform(addlog(imp.transform(X))))
    return f

# --- per-host sliding windows; shuffle_within destroys temporal order but
#     keeps window count / capacity / label mix identical (the E1 control) ----
def _windows(X, y, meta, T, stride=1, shuffle_within=False, seed=42):
    nf = X.shape[1]
    if T == 1:
        return X.reshape(-1, 1, nf).astype(np.float32), y.astype(np.int32)
    Xw, yw = [], []
    m = meta.reset_index(drop=True)
    rng = np.random.default_rng(seed)
    for _, idx in m.groupby("group").groups.items():
        idx = np.asarray(idx)
        idx = idx[np.argsort(m.loc[idx, "ts"].values, kind="stable")]
        if shuffle_within:
            idx = idx.copy(); rng.shuffle(idx)
        gX, gy, n = X[idx], y[idx], len(idx)
        if n < T:
            pad = np.repeat(gX[:1], T - n, axis=0)
            Xw.append(np.concatenate([pad, gX])[None]); yw.append(gy[-1]); continue
        for s in range(0, n - T + 1, stride):
            Xw.append(gX[s:s + T][None]); yw.append(gy[s + T - 1])
    return np.concatenate(Xw, 0).astype(np.float32), np.asarray(yw, np.int32)

def _chrono_idx(meta, tr=0.70, va=0.15):
    o = np.argsort(meta["ts"].values, kind="stable")
    n = len(o); ntr = int(n * tr); nva = int(n * va)
    return o[:ntr], o[ntr:ntr + nva], o[ntr + nva:]

def _theta_for(dataset):
    return (load_results(dataset).get("sboa", {}).get("theta_star")
            or DEFAULT_THETA_BY_DATASET.get(dataset,
                 {"bigru1": 128, "bigru2": 64, "lr": LR, "dropout": DROPOUT,
                  "optimizer": "adam", "top_k": TOP_K}))

def _train_eval_windows(Xtr, ytr, Xva, yva, Xte, yte, ncls, theta, T, epochs, seed):
    nf = Xtr.shape[2]
    corr = compute_corr_matrix(Xtr.reshape(-1, nf))
    A = adjacency_from_corr(corr, theta["top_k"])
    tf.keras.backend.clear_session(); np.random.seed(seed); tf.random.set_seed(seed)
    model = build_gcn_bigru(nf, ncls, A, T, dropout=theta["dropout"], reg=REG,
                            bigru1=theta["bigru1"], bigru2=theta["bigru2"],
                            top_k=theta["top_k"], adaptive=ADAPTIVE_GRAPH)
    opt = OPTIMIZERS[theta["optimizer"]](learning_rate=theta["lr"], clipnorm=1.0)
    model.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    freq = np.bincount(ytr, minlength=ncls).astype(float) / max(len(ytr), 1)
    cw = np.clip((1.0 / np.sqrt(freq + 1e-8)) / (1.0 / np.sqrt(freq + 1e-8)).mean(), 0.5, 15.0)
    es = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True,
                       mode="min", verbose=0)
    t0 = _t_time.perf_counter()
    model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=epochs, batch_size=BATCH,
              class_weight=dict(enumerate(cw)), callbacks=[es], verbose=0)
    dt = _t_time.perf_counter() - t0
    pred = np.argmax(model.predict(Xte, batch_size=BATCH, verbose=0), 1)
    acc = float(accuracy_score(yte, pred) * 100)
    f1 = float(f1_score(yte, pred, average="weighted", zero_division=0) * 100)
    return acc, f1, dt

# === E1 + E2: chronological, per-host sliding windows ========================
def run_temporal_validation(dataset, T_values=None, epochs=None, cap=None, seed=SEED):
    T_values = T_values or TEMPORAL_T_VALUES
    epochs = epochs or TEMPORAL_EPOCHS
    cap = TEMPORAL_ROW_CAP if cap is None else cap
    loaded = temporal_load(dataset)
    if loaded is None:
        print(f"[{dataset}] no timestamp metadata -> skipped (keep T=1, state in paper)")
        return None
    X, y, class_order, meta = loaded
    X, y, meta = _cap_by_groups(X, y, meta, cap, seed)
    ncls = len(class_order)
    itr, iva, ite = _chrono_idx(meta)
    prep = _make_prep(X[itr])
    Xtr_f, Xva_f, Xte_f = prep(X[itr]), prep(X[iva]), prep(X[ite])
    mtr = meta.iloc[itr].reset_index(drop=True)
    mva = meta.iloc[iva].reset_index(drop=True)
    mte = meta.iloc[ite].reset_index(drop=True)
    ytr, yva, yte = y[itr], y[iva], y[ite]
    theta = _theta_for(dataset)
    print(f"[{dataset}] temporal validation: rows train/val/test = "
          f"{len(itr):,}/{len(iva):,}/{len(ite):,} | hosts={meta['group'].nunique():,}")
    rows = []
    for T in T_values:
        modes = ["temporal"] if T == 1 else ["temporal", "shuffled"]
        for mode in modes:
            sh = (mode == "shuffled")
            Xa, ya = _windows(Xtr_f, ytr, mtr, T, 1, sh, seed)
            Xb, yb = _windows(Xva_f, yva, mva, T, 1, sh, seed)
            Xc, yc = _windows(Xte_f, yte, mte, T, 1, sh, seed)
            acc, f1, dt = _train_eval_windows(Xa, ya, Xb, yb, Xc, yc, ncls, theta, T, epochs, seed)
            rows.append({"dataset": dataset, "timesteps": int(T), "order": mode,
                         "test_accuracy": acc, "test_f1": f1, "train_time_sec": dt,
                         "n_train_seq": int(len(Xa)), "n_test_seq": int(len(Xc))})
            print(f"  T={T:2d} [{mode:9s}] acc={acc:.2f}% f1={f1:.2f}% "
                  f"n_train_seq={len(Xa):,} ({dt:.1f}s)")
            with open(RESULTS_DIR / f"temporal_validation_{dataset}.json", "w") as fh:
                _json.dump(rows, fh, indent=2, default=float)
    return pd.DataFrame(rows)

# === E3: non-overlapping windows + window-level shuffle split =================
# Apples-to-apples with the original shuffled-split headline: no row shared
# between windows (stride=T) so a window-level stratified shuffle split has no
# leakage, while matching the original split *style*.
def run_temporal_headline_check(dataset, T=None, epochs=None, cap=None, seed=SEED):
    T = T or TEMPORAL_HEADLINE_T
    epochs = epochs or TEMPORAL_EPOCHS
    cap = TEMPORAL_ROW_CAP if cap is None else cap
    loaded = temporal_load(dataset)
    if loaded is None:
        print(f"[{dataset}] no timestamp metadata -> headline check skipped")
        return None
    X, y, class_order, meta = loaded
    X, y, meta = _cap_by_groups(X, y, meta, cap, seed)
    ncls = len(class_order)
    # build non-overlapping RAW windows per host (features scaled later, per fold)
    def _nonoverlap_raw(Xr, yr, m, T):
        Xw, yw = [], []
        mm = m.reset_index(drop=True)
        for _, idx in mm.groupby("group").groups.items():
            idx = np.asarray(idx)
            idx = idx[np.argsort(mm.loc[idx, "ts"].values, kind="stable")]
            n = len(idx)
            for s in range(0, n - T + 1, T):
                Xw.append(Xr[idx[s:s + T]][None]); yw.append(yr[idx[s + T - 1]])
        if not Xw:
            return np.empty((0, T, Xr.shape[1]), np.float32), np.empty((0,), np.int32)
        return np.concatenate(Xw, 0).astype(np.float32), np.asarray(yw, np.int32)

    Xw, yw = _nonoverlap_raw(X, y, meta, T)           # raw (unscaled) windows
    idx = np.arange(len(Xw))
    tr, te = train_test_split(idx, test_size=0.15, random_state=seed, stratify=yw)
    tr, va = train_test_split(tr, test_size=0.176, random_state=seed, stratify=yw[tr])
    nf0 = Xw.shape[2]
    prep = _make_prep(Xw[tr].reshape(-1, nf0))          # fit on train-window rows only
    def _scale(A):
        return prep(A.reshape(-1, nf0)).reshape(A.shape[0], T, -1)
    Xtr, Xva, Xte = _scale(Xw[tr]), _scale(Xw[va]), _scale(Xw[te])
    theta = _theta_for(dataset)
    accT, f1T, _ = _train_eval_windows(Xtr, yw[tr], Xva, yw[va], Xte, yw[te],
                                       ncls, theta, T, epochs, seed)
    # T=1 comparison on the SAME window-shuffle split style
    X1 = X.reshape(-1, 1, X.shape[1])
    i1 = np.arange(len(X1))
    tr1, te1 = train_test_split(i1, test_size=0.15, random_state=seed, stratify=y)
    tr1, va1 = train_test_split(tr1, test_size=0.176, random_state=seed, stratify=y[tr1])
    prep1 = _make_prep(X[tr1])
    s1 = lambda A: prep1(A.reshape(-1, X.shape[1])).reshape(A.shape[0], 1, -1)
    acc1, f11, _ = _train_eval_windows(s1(X1[tr1]), y[tr1], s1(X1[va1]), y[va1],
                                       s1(X1[te1]), y[te1], ncls, theta, 1, epochs, seed)
    out = {"dataset": dataset, "T": int(T),
           "T1_accuracy": acc1, "T1_f1": f11,
           "Twin_accuracy": accT, "Twin_f1": f1T,
           "delta_accuracy": accT - acc1, "delta_f1": f1T - f11}
    print(f"[{dataset}] headline check (window-shuffle split): "
          f"T=1 f1={f11:.2f}% vs T={T} f1={f1T:.2f}% (delta={f1T - f11:+.2f}pp)")
    with open(RESULTS_DIR / f"temporal_headline_{dataset}.json", "w") as fh:
        _json.dump(out, fh, indent=2, default=float)
    return out


In [ ]:
# === Run the temporal-validation experiments =================================
if RUN_TEMPORAL_VALIDATION:
    _temporal_frames = {}
    for _ds in TEMPORAL_DATASETS:
        try:
            _df = run_temporal_validation(_ds)
            if _df is not None:
                _temporal_frames[_ds] = _df
        except FileNotFoundError as _e:
            print(f"[{_ds}] dataset files not found -> skipped ({_e})")
        except Exception as _e:
            print(f"[{_ds}] temporal validation failed: {_e}")

    # ---- plot: E1/E2 temporal vs shuffled-control, per dataset ----
    if _temporal_frames:
        n = len(_temporal_frames)
        fig, axes = plt.subplots(1, n, figsize=(6 * n, 4.5), squeeze=False)
        for ax, (ds, df) in zip(axes[0], _temporal_frames.items()):
            tmp = df[df["order"] == "temporal"].sort_values("timesteps")
            shf = df[df["order"] == "shuffled"].sort_values("timesteps")
            ax.plot(tmp["timesteps"], tmp["test_f1"], marker="o", label="temporal order")
            if len(shf):
                ax.plot(shf["timesteps"], shf["test_f1"], marker="s", ls="--",
                        label="shuffled control")
            base = tmp[tmp["timesteps"] == 1]["test_f1"]
            if len(base):
                ax.axhline(float(base.iloc[0]), color="gray", ls=":", label="T=1 baseline")
            ax.set_title(f"{ds}", fontweight="bold")
            ax.set_xlabel("timesteps (T)"); ax.set_ylabel("test F1 (%)")
            ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
        fig.suptitle("Temporal windowing: real order vs destroyed-order control",
                     fontweight="bold")
        plt.tight_layout()
        plt.savefig("temporal_validation.png", dpi=150, bbox_inches="tight")
        plt.show()

        # ---- verdict per dataset ----
        print("\n=== Temporal-signal verdict (E1/E2) ===")
        for ds, df in _temporal_frames.items():
            t = df[df["order"] == "temporal"]
            s = df[df["order"] == "shuffled"]
            base = float(t[t["timesteps"] == 1]["test_f1"].iloc[0]) if (t["timesteps"] == 1).any() else np.nan
            best_row = t[t["timesteps"] > 1].sort_values("test_f1").tail(1)
            if len(best_row):
                bt = int(best_row["timesteps"].iloc[0]); bf1 = float(best_row["test_f1"].iloc[0])
                sf1 = s[s["timesteps"] == bt]["test_f1"]
                sf1 = float(sf1.iloc[0]) if len(sf1) else np.nan
                gain_vs_base = bf1 - base
                gain_vs_ctrl = bf1 - sf1
                verdict = ("TEMPORAL SIGNAL CONFIRMED"
                           if (gain_vs_base > 0.3 and gain_vs_ctrl > 0.3)
                           else "no clear temporal gain")
                print(f"  {ds:10s}: best T={bt:2d} f1={bf1:.2f}%  "
                      f"(vs T=1 {gain_vs_base:+.2f}pp, vs shuffled-control {gain_vs_ctrl:+.2f}pp) "
                      f"-> {verdict}")

    # ---- E3 apples-to-apples headline check ----
    if RUN_TEMPORAL_HEADLINE:
        print("\n=== E3: window-shuffle-split headline check ===")
        for _ds in TEMPORAL_DATASETS:
            try:
                run_temporal_headline_check(_ds)
            except FileNotFoundError as _e:
                print(f"[{_ds}] dataset files not found -> skipped ({_e})")
            except Exception as _e:
                print(f"[{_ds}] headline check failed: {_e}")
